# Team 23 — Part 2

Canonical modified supplied Transformer notebook.

Team 23 submission metadata is supplied privately at delivery time. Completed contributions and official membership must be confirmed by the team; no personal names, SRNs, or contribution assignments are stored in this shared repository.

In [ ]:
# Team 23 standalone runtime bootstrap; source bundle SHA256=591943802b6e80238ba93a101b8620a1aac55ad3e26888e05b85d93012451f33
import base64, csv, hashlib, json, os, platform, shutil, sys, tempfile
from pathlib import Path
SOURCE_BUNDLE = {'scripts/weight_baselines.py': '#!/usr/bin/env python3\n"""Leakage-aware scalar baselines for the paired Team 23 weight arrays.\n\nThe two equal-length halves are treated as corresponding offset blocks only for\nthe purpose of conservative split assignment. This script does not infer that\nthey are checkpoints or otherwise equivalent tensors.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\nfrom typing import Literal\n\nimport numpy as np\n\n\nSplitName = Literal["fitting", "development", "audit", "guard"]\nSPLIT_NAMES: tuple[str, ...] = ("fitting", "development", "audit")\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _group_name(seed: int, group_id: int) -> str:\n    value = hashlib.sha256(f"weight-v1|{seed}|{group_id}".encode("ascii")).digest()[0] / 256.0\n    if value < 0.60:\n        return "fitting"\n    if value < 0.80:\n        return "development"\n    return "audit"\n\n\ndef make_assignment(\n    total_length: int,\n    *,\n    half_length: int,\n    seed: int = 2301,\n    block_size: int = 4096,\n    guard: int = 32,\n) -> np.ndarray:\n    """Return one label per scalar, with -1 for guard values.\n\n    Assignment is based only on positions, lengths, and the fixed seed. The\n    same group ID is used at the same offset in both halves.\n    """\n    if total_length != 2 * half_length or total_length <= 0:\n        raise ValueError("expected two equal-length halves")\n    if block_size <= 0 or guard < 0:\n        raise ValueError("block_size must be positive and guard must be nonnegative")\n    labels = np.full(total_length, -1, dtype=np.int8)\n    codes = {"fitting": 0, "development": 1, "audit": 2}\n    group_count = (half_length + block_size - 1) // block_size\n    group_labels = [_group_name(seed, group_id) for group_id in range(group_count)]\n    for half_start in (0, half_length):\n        for group_id, name in enumerate(group_labels):\n            start = half_start + group_id * block_size\n            stop = min(half_start + (group_id + 1) * block_size, half_start + half_length)\n            labels[start:stop] = codes[name]\n    if guard:\n        for group_id in range(1, group_count):\n            if group_labels[group_id] != group_labels[group_id - 1]:\n                boundary_offsets = (group_id * block_size, half_length + group_id * block_size)\n                for boundary in boundary_offsets:\n                    labels[max(0, boundary - guard) : min(total_length, boundary + guard)] = -1\n    return labels\n\n\ndef validate_assignment(labels: np.ndarray) -> None:\n    labels = np.asarray(labels)\n    if labels.ndim != 1 or not np.isin(labels, [-1, 0, 1, 2]).all():\n        raise AssertionError("assignment contains invalid or overlapping labels")\n    if not np.all((labels >= -1) & (labels <= 2)):\n        raise AssertionError("assignment has an invalid label")\n    if np.any(labels == -1):\n        # Guards are the only permitted unassigned positions.\n        return\n\n\ndef fit_coefficients(noisy: np.ndarray, clean: np.ndarray, labels: np.ndarray) -> dict[str, float]:\n    fit = labels == 0\n    if not np.any(fit):\n        raise ValueError("fitting split is empty")\n    x = np.asarray(noisy[fit], dtype=np.float64)\n    y = np.asarray(clean[fit], dtype=np.float64)\n    residual = y - x\n    bias = float(np.mean(residual))\n    x_centered = x - np.mean(x)\n    variance = float(np.mean(x_centered * x_centered))\n    slope = float(np.mean(x_centered * (y - np.mean(y))) / variance) if variance else 1.0\n    intercept = float(np.mean(y) - slope * np.mean(x))\n    return {\n        "identity_slope": 1.0,\n        "identity_intercept": 0.0,\n        "bias_residual_mean": bias,\n        "bias_prediction_slope": 1.0,\n        "bias_prediction_intercept": bias,\n        "affine_prediction_slope": slope,\n        "affine_prediction_intercept": intercept,\n    }\n\n\ndef rmse(prediction: np.ndarray, clean: np.ndarray, labels: np.ndarray, code: int) -> float:\n    dev = labels == code\n    if not np.any(dev):\n        raise ValueError("development split is empty")\n    error = np.asarray(prediction[dev], dtype=np.float64) - np.asarray(clean[dev], dtype=np.float64)\n    return float(np.sqrt(np.mean(error * error)))\n\n\ndef evaluate(noisy: np.ndarray, clean: np.ndarray, labels: np.ndarray, coefficients: dict[str, float]) -> dict[str, float]:\n    x = np.asarray(noisy)\n    identity = x\n    bias = x + coefficients["bias_prediction_intercept"]\n    affine = coefficients["affine_prediction_slope"] * x + coefficients["affine_prediction_intercept"]\n    return {\n        "identity": rmse(identity, clean, labels, 1),\n        "bias_correction": rmse(bias, clean, labels, 1),\n        "global_affine": rmse(affine, clean, labels, 1),\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data-dir", type=Path, default=Path("data/23_Team_Toxic"))\n    parser.add_argument("--output", type=Path, default=Path("results/baselines/weight_v1.json"))\n    parser.add_argument("--seed", type=int, default=2301)\n    parser.add_argument("--block-size", type=int, default=4096)\n    parser.add_argument("--guard", type=int, default=32)\n    args = parser.parse_args()\n    data_dir = args.data_dir\n    clean_path = data_dir / "train_weights_clean.npy"\n    noisy_path = data_dir / "train_weights_noisy.npy"\n    clean = np.load(clean_path, mmap_mode="r")\n    noisy = np.load(noisy_path, mmap_mode="r")\n    if clean.shape != noisy.shape or clean.ndim != 1 or not np.isfinite(clean).all() or not np.isfinite(noisy).all():\n        raise ValueError("clean/noisy arrays must be finite, one-dimensional, and shape-aligned")\n    if clean.size % 2:\n        raise ValueError("training array length must have two equal halves")\n    # Freeze position-only assignment before any clean labels enter a fit.\n    labels = make_assignment(\n        int(clean.size), half_length=int(clean.size // 2), seed=args.seed,\n        block_size=args.block_size, guard=args.guard,\n    )\n    validate_assignment(labels)\n    coefficients = fit_coefficients(noisy, clean, labels)\n    metrics = evaluate(noisy, clean, labels, coefficients)\n    split_counts = {\n        "fitting": int(np.count_nonzero(labels == 0)),\n        "development": int(np.count_nonzero(labels == 1)),\n        "audit": int(np.count_nonzero(labels == 2)),\n        "guard": int(np.count_nonzero(labels == -1)),\n    }\n    split_spec = {\n        "rule": "sha256(weight-v1|seed|group_id) byte thresholds 0.60/0.80; same offset group in both halves",\n        "seed": args.seed,\n        "block_size": args.block_size,\n        "guard": args.guard,\n        "half_length": int(clean.size // 2),\n        "total_length": int(clean.size),\n    }\n    split_hash = hashlib.sha256(json.dumps(split_spec, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n    result = {\n        "schema_version": 1,\n        "experiment": "BASE-WEIGHT-001",\n        "team": 23,\n        "python": platform.python_version(),\n        "numpy": np.__version__,\n        "split": {**split_spec, "spec_sha256": split_hash, "counts": split_counts},\n        "input_sha256": {name: sha256(data_dir / name) for name in ("train_weights_clean.npy", "train_weights_noisy.npy")},\n        "fit_coefficients": coefficients,\n        "development_rmse_original_units": metrics,\n        "audit_metrics": None,\n        "limitations": [\n            "The clean audit labels are reserved and no audit-label metric is computed.",\n            "The equal-length halves are grouped conservatively by offset only; their checkpoint meaning is unverified.",\n            "Scalar baselines do not establish performance on the clean test weights, whose labels are unavailable.",\n        ],\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    print(json.dumps({"output": str(args.output), "development_rmse_original_units": metrics, "counts": split_counts}, sort_keys=True))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/train_scalar_mlp.py': '#!/usr/bin/env python3\n"""Train a bounded scalar residual MLP on the fixed weight split.\n\nThe CLI is intentionally conservative: it reads only fitting and development\nrows from the clean array, never reads clean audit rows, and writes checkpoints\natomically.  The module API is used by synthetic tests and by future bounded\njobs; this file does not launch a job when imported.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport random\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\n\nfrom scripts.weight_baselines import make_assignment\n\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / "data" / "23_Team_Toxic"\nDEFAULT_CLEAN = DATA / "train_weights_clean.npy"\nDEFAULT_NOISY = DATA / "train_weights_noisy.npy"\nDEFAULT_CHECKPOINT = ROOT / "results" / "checkpoints" / "scalar_mlp.pt"\n# v1 optimized a broadcast B-by-B loss rather than aligned scalar pairs.\n# Its optimizer trajectory cannot be resumed under the corrected objective.\nSCHEMA_VERSION = 2\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef split_spec(total_length: int, *, seed: int = 2301, block_size: int = 4096, guard: int = 32) -> dict[str, Any]:\n    if total_length <= 0 or total_length % 2:\n        raise ValueError("weight length must be positive and evenly divisible into two halves")\n    spec = {\n        "rule": "sha256(weight-v1|seed|group_id) byte thresholds 0.60/0.80; same offset group in both halves",\n        "seed": int(seed),\n        "block_size": int(block_size),\n        "guard": int(guard),\n        "half_length": int(total_length // 2),\n        "total_length": int(total_length),\n    }\n    encoded = json.dumps(spec, sort_keys=True, separators=(",", ":")).encode("utf-8")\n    spec["spec_sha256"] = hashlib.sha256(encoded).hexdigest()\n    return spec\n\n\ndef split_labels(total_length: int, *, seed: int = 2301, block_size: int = 4096, guard: int = 32) -> tuple[np.ndarray, dict[str, Any]]:\n    spec = split_spec(total_length, seed=seed, block_size=block_size, guard=guard)\n    labels = make_assignment(total_length, half_length=total_length // 2, seed=seed, block_size=block_size, guard=guard)\n    counts = {name: int(np.count_nonzero(labels == code)) for name, code in (("fitting", 0), ("development", 1), ("audit", 2), ("guard", -1))}\n    return labels, {**spec, "counts": counts}\n\n\nclass ScalarResidualMLP(torch.nn.Module):\n    """Small scalar-to-scalar residual network."""\n\n    def __init__(self, hidden: int = 32):\n        super().__init__()\n        if hidden <= 0:\n            raise ValueError("hidden must be positive")\n        self.network = torch.nn.Sequential(\n            torch.nn.Linear(1, hidden),\n            torch.nn.Tanh(),\n            torch.nn.Linear(hidden, 1),\n        )\n\n    def forward(self, noisy: torch.Tensor) -> torch.Tensor:\n        values = noisy.reshape(-1, 1)\n        return (values + self.network(values)).reshape_as(noisy)\n\n\ndef paired_mse(prediction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n    """Refuse silent pairwise broadcasting in the scalar regression loss."""\n    if prediction.shape != target.shape or prediction.numel() == 0:\n        raise ValueError("scalar prediction and target shapes must match exactly and be nonempty")\n    loss = torch.mean((prediction - target) ** 2)\n    if not bool(torch.isfinite(loss)):\n        raise FloatingPointError("non-finite scalar regression loss")\n    return loss\n\n\ndef _rng_state() -> dict[str, Any]:\n    state: dict[str, Any] = {\n        "python": random.getstate(),\n        "numpy": np.random.get_state(),\n        "torch": torch.get_rng_state(),\n    }\n    if torch.backends.mps.is_available() and hasattr(torch.mps, "get_rng_state"):\n        state["mps"] = torch.mps.get_rng_state()\n    return state\n\n\ndef _restore_rng(state: dict[str, Any]) -> None:\n    required = {"python", "numpy", "torch"}\n    if not required.issubset(state):\n        raise ValueError("checkpoint is missing a required RNG state")\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch"])\n    if "mps" in state:\n        if not (torch.backends.mps.is_available() and hasattr(torch.mps, "set_rng_state")):\n            raise ValueError("checkpoint contains MPS RNG state but MPS is unavailable")\n        torch.mps.set_rng_state(state["mps"])\n\n\ndef _atomic_torch_save(payload: dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary: Path | None = None\n    try:\n        with tempfile.NamedTemporaryFile(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n        torch.save(payload, temporary)\n        os.replace(temporary, path)\n    finally:\n        if temporary is not None and temporary.exists():\n            temporary.unlink()\n\n\ndef _validate_arrays(noisy: np.ndarray, clean: np.ndarray, labels: np.ndarray) -> None:\n    if noisy.ndim != 1 or clean.ndim != 1 or labels.ndim != 1 or noisy.shape != clean.shape or noisy.shape != labels.shape:\n        raise ValueError("noisy, clean, and labels must be aligned one-dimensional arrays")\n    if noisy.dtype != np.float32 or clean.dtype != np.float32:\n        raise ValueError("noisy and clean arrays must be float32")\n    if not np.isin(labels, [-1, 0, 1, 2]).all():\n        raise ValueError("split labels must be fitting, development, audit, or guard")\n    # Preflight validates the complete files. This trainer must not inspect\n    # clean audit values, even for its own finite-value gate.\n    allowed = (labels == 0) | (labels == 1)\n    if not np.isfinite(noisy).all() or not np.isfinite(clean[allowed]).all():\n        raise ValueError("noisy and fitting/development clean values must be finite")\n\n\ndef _checkpoint_payload(\n    model: ScalarResidualMLP,\n    optimizer: torch.optim.Optimizer,\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\n    *,\n    epoch: int,\n    step: int,\n    config: dict[str, Any],\n    input_hashes: dict[str, str],\n    split: dict[str, Any],\n) -> dict[str, Any]:\n    return {\n        "schema_version": SCHEMA_VERSION,\n        "model": model.state_dict(),\n        "optimizer": optimizer.state_dict(),\n        "scheduler": scheduler.state_dict(),\n        "epoch": int(epoch),\n        "step": int(step),\n        "config": config,\n        "input_sha256": input_hashes,\n        "split": split,\n        "rng": _rng_state(),\n    }\n\n\ndef _validate_checkpoint(payload: dict[str, Any], expected: dict[str, Any]) -> None:\n    required = {"schema_version", "model", "optimizer", "scheduler", "epoch", "step", "config", "input_sha256", "split", "rng"}\n    if not required.issubset(payload) or payload["schema_version"] != SCHEMA_VERSION:\n        raise ValueError("checkpoint schema is incomplete or unsupported")\n    for key in ("config", "input_sha256", "split"):\n        if payload[key] != expected[key]:\n            raise ValueError(f"checkpoint {key} does not match the current run")\n    if not isinstance(payload["epoch"], int) or not isinstance(payload["step"], int) or payload["epoch"] < 0 or payload["step"] < 0:\n        raise ValueError("checkpoint progress counters are invalid")\n    _restore_rng(payload["rng"])\n\n\ndef _load_checkpoint(path: Path, expected: dict[str, Any], model: ScalarResidualMLP, optimizer: torch.optim.Optimizer, scheduler: torch.optim.lr_scheduler.LRScheduler) -> tuple[int, int]:\n    try:\n        payload = torch.load(path, map_location="cpu", weights_only=False)\n    except Exception as exc:\n        raise ValueError(f"unable to read checkpoint: {path}") from exc\n    if not isinstance(payload, dict):\n        raise ValueError("checkpoint root must be a mapping")\n    _validate_checkpoint(payload, expected)\n    try:\n        model.load_state_dict(payload["model"], strict=True)\n        optimizer.load_state_dict(payload["optimizer"])\n        scheduler.load_state_dict(payload["scheduler"])\n    except Exception as exc:\n        raise ValueError("checkpoint model or optimizer state is incompatible") from exc\n    return int(payload["epoch"]), int(payload["step"])\n\n\ndef _device(name: str | None) -> torch.device:\n    selected = name or ("mps" if torch.backends.mps.is_available() else "cpu")\n    device = torch.device(selected)\n    if device.type == "mps" and not torch.backends.mps.is_available():\n        raise ValueError("MPS was requested but is unavailable")\n    return device\n\n\ndef _rmse(model: ScalarResidualMLP, noisy: np.ndarray, clean: np.ndarray, indices: np.ndarray, device: torch.device, chunk_size: int) -> float:\n    if indices.size == 0:\n        raise ValueError("development split is empty")\n    model.eval()\n    squared = 0.0\n    count = 0\n    with torch.no_grad():\n        for start in range(0, indices.size, chunk_size):\n            batch = indices[start : start + chunk_size]\n            x = torch.from_numpy(np.asarray(noisy[batch])).to(device=device, dtype=torch.float32)\n            prediction = model(x).detach().cpu().numpy().reshape(-1).astype(np.float64)\n            error = prediction - np.asarray(clean[batch], dtype=np.float64)\n            squared += float(np.dot(error, error))\n            count += error.size\n    return float(np.sqrt(squared / count))\n\n\ndef train(\n    noisy: np.ndarray,\n    clean: np.ndarray,\n    labels: np.ndarray,\n    *,\n    config: dict[str, Any] | None = None,\n    checkpoint: Path | None = None,\n    input_hashes: dict[str, str] | None = None,\n    resume: bool = False,\n    device_name: str | None = None,\n    max_epochs_this_run: int | None = None,\n) -> dict[str, Any]:\n    """Train for the bounded config and return development-only metrics.\n\n    ``clean`` is indexed only with fitting/development masks.  A caller may\n    provide file hashes without loading complete arrays into memory.\n    """\n    _validate_arrays(noisy, clean, labels)\n    settings = {\n        "hidden": 32,\n        "learning_rate": 1e-3,\n        "weight_decay": 0.0,\n        "batch_size": 4096,\n        "epochs": 2,\n        "chunk_size": 65536,\n        "seed": 2301,\n    }\n    if config:\n        settings.update(config)\n    if any(int(settings[key]) <= 0 for key in ("hidden", "batch_size", "epochs", "chunk_size")) or float(settings["learning_rate"]) <= 0:\n        raise ValueError("training sizes and learning_rate must be positive")\n    if max_epochs_this_run is not None and max_epochs_this_run <= 0:\n        raise ValueError("max_epochs_this_run must be positive")\n    labels = np.asarray(labels)\n    fit_indices = np.flatnonzero(labels == 0)\n    dev_indices = np.flatnonzero(labels == 1)\n    if fit_indices.size == 0 or dev_indices.size == 0:\n        raise ValueError("fitting and development splits must be non-empty")\n    split = {\n        **split_spec(int(labels.size)),\n        "counts": {"fitting": int(fit_indices.size), "development": int(dev_indices.size), "audit": int(np.count_nonzero(labels == 2)), "guard": int(np.count_nonzero(labels == -1))},\n        "labels_sha256": hashlib.sha256(labels.tobytes()).hexdigest(),\n    }\n    expected = {"config": settings, "input_sha256": dict(input_hashes or {}), "split": split}\n    device = _device(device_name)\n    random.seed(int(settings["seed"]))\n    np.random.seed(int(settings["seed"]))\n    torch.manual_seed(int(settings["seed"]))\n    model = ScalarResidualMLP(int(settings["hidden"])).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=float(settings["learning_rate"]), weight_decay=float(settings["weight_decay"]))\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(settings["epochs"]))\n    start_epoch = 0\n    step = 0\n    if resume:\n        if checkpoint is None or not checkpoint.is_file():\n            raise ValueError("resume requires an existing checkpoint")\n        start_epoch, step = _load_checkpoint(checkpoint, expected, model, optimizer, scheduler)\n    stop_epoch = min(int(settings["epochs"]), start_epoch + max_epochs_this_run) if max_epochs_this_run else int(settings["epochs"])\n    for epoch in range(start_epoch, stop_epoch):\n        model.train()\n        # Epoch-specific permutations make interrupted and uninterrupted runs identical.\n        order = np.random.default_rng(int(settings["seed"]) + epoch).permutation(fit_indices.size)\n        for start in range(0, fit_indices.size, int(settings["batch_size"])):\n            positions = order[start : start + int(settings["batch_size"])]\n            indices = fit_indices[positions]\n            x = torch.from_numpy(np.asarray(noisy[indices])).to(device=device, dtype=torch.float32)\n            y = torch.from_numpy(np.asarray(clean[indices])).to(device=device, dtype=torch.float32)\n            optimizer.zero_grad(set_to_none=True)\n            loss = paired_mse(model(x), y)\n            loss.backward()\n            optimizer.step()\n            step += 1\n        scheduler.step()\n        if checkpoint is not None:\n            _atomic_torch_save(_checkpoint_payload(model, optimizer, scheduler, epoch=epoch + 1, step=step, config=settings, input_hashes=dict(input_hashes or {}), split=split), checkpoint)\n    return {"development_rmse_original_units": _rmse(model, noisy, clean, dev_indices, device, int(settings["chunk_size"])), "epoch": stop_epoch, "step": step, "device": str(device), "audit_metrics": None}\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--clean", type=Path, default=DEFAULT_CLEAN)\n    parser.add_argument("--noisy", type=Path, default=DEFAULT_NOISY)\n    parser.add_argument("--checkpoint", type=Path, default=DEFAULT_CHECKPOINT)\n    parser.add_argument("--resume", action="store_true")\n    parser.add_argument("--epochs", type=int, default=2)\n    parser.add_argument("--batch-size", type=int, default=4096)\n    parser.add_argument("--hidden", type=int, default=32)\n    parser.add_argument("--device")\n    parser.add_argument("--max-epochs-this-run", type=int, default=1)\n    args = parser.parse_args()\n    noisy = np.load(args.noisy, mmap_mode="r", allow_pickle=False)\n    clean = np.load(args.clean, mmap_mode="r", allow_pickle=False)\n    labels, split = split_labels(int(noisy.size))\n    hashes = {"clean": sha256_file(args.clean), "noisy": sha256_file(args.noisy)}\n    config = {"epochs": args.epochs, "batch_size": args.batch_size, "hidden": args.hidden}\n    result = train(noisy.reshape(-1), clean.reshape(-1), labels, config=config, checkpoint=args.checkpoint, input_hashes=hashes, resume=args.resume, device_name=args.device, max_epochs_this_run=args.max_epochs_this_run)\n    print(json.dumps({"checkpoint": str(args.checkpoint), "split": split, **result}, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'scripts/export_scalar_weights.py': '#!/usr/bin/env python3\n"""Export a schema2 scalar MLP checkpoint as verified Part 1 artifacts."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\n\nfrom scripts.train_scalar_mlp import SCHEMA_VERSION, ScalarResidualMLP, _validate_checkpoint, sha256_file, split_labels\n\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_DATA = ROOT / "data" / "23_Team_Toxic"\nDEFAULT_CHECKPOINT = ROOT / "results" / "checkpoints" / "scalar_mlp.pt"\nDEFAULT_NPY = ROOT / "results" / "candidates" / "scalar_mlp_restored.npy"\nDEFAULT_CSV = ROOT / "results" / "candidates" / "scalar_mlp_submission.csv"\nEXPECTED_TEST_COUNT = 2_961_630\nHASH_LENGTH = 64\nFIXED_SEED = 2301\nFIXED_BLOCK_SIZE = 4096\nFIXED_GUARD = 32\n\n\ndef _finite_tree(value: Any, name: str) -> None:\n    if isinstance(value, torch.Tensor):\n        if not bool(torch.isfinite(value).all()):\n            raise ValueError(f"checkpoint contains non-finite value in {name}")\n        return\n    if isinstance(value, np.ndarray):\n        if np.issubdtype(value.dtype, np.number) and not np.isfinite(value).all():\n            raise ValueError(f"checkpoint contains non-finite value in {name}")\n        return\n    if isinstance(value, (float, np.floating)) and not np.isfinite(value):\n        raise ValueError(f"checkpoint contains non-finite value in {name}")\n    if isinstance(value, dict):\n        for key, child in value.items():\n            _finite_tree(child, f"{name}.{key}")\n    elif isinstance(value, (list, tuple)):\n        for index, child in enumerate(value):\n            _finite_tree(child, f"{name}[{index}]")\n\n\ndef _hash(path: Path) -> str:\n    return sha256_file(path)\n\n\ndef _validate_hashes(payload: dict[str, Any], data_dir: Path) -> dict[str, str]:\n    hashes = payload.get("input_sha256")\n    if not isinstance(hashes, dict) or set(hashes) != {"clean", "noisy"}:\n        raise ValueError("checkpoint input hashes are missing or incomplete")\n    for key, digest in hashes.items():\n        if not isinstance(digest, str) or len(digest) != HASH_LENGTH or any(c not in "0123456789abcdef" for c in digest):\n            raise ValueError(f"checkpoint input hash is invalid: {key}")\n        path = data_dir / ("train_weights_clean.npy" if key == "clean" else "train_weights_noisy.npy")\n        if not path.is_file() or _hash(path) != digest:\n            raise ValueError(f"checkpoint input hash does not match local file: {key}")\n    return {key: str(value) for key, value in hashes.items()}\n\n\ndef _validate_config(config: Any) -> dict[str, Any]:\n    if not isinstance(config, dict):\n        raise ValueError("checkpoint config is missing or incompatible")\n    required = {"hidden", "learning_rate", "weight_decay", "batch_size", "epochs", "chunk_size", "seed"}\n    if set(config) != required:\n        raise ValueError("checkpoint config is missing or incompatible")\n    integer_keys = {"hidden", "batch_size", "epochs", "chunk_size", "seed"}\n    for key in integer_keys:\n        if type(config[key]) is not int:\n            raise ValueError("checkpoint config is missing or incompatible")\n    if any(config[key] <= 0 for key in ("hidden", "batch_size", "epochs", "chunk_size")):\n        raise ValueError("checkpoint config is missing or incompatible")\n    if type(config["learning_rate"]) not in (int, float) or not np.isfinite(config["learning_rate"]) or config["learning_rate"] <= 0:\n        raise ValueError("checkpoint config is missing or incompatible")\n    if type(config["weight_decay"]) not in (int, float) or not np.isfinite(config["weight_decay"]) or config["weight_decay"] < 0:\n        raise ValueError("checkpoint config is missing or incompatible")\n    return dict(config)\n\n\ndef _validate_split(payload: dict[str, Any], total_length: int) -> dict[str, Any]:\n    split = payload.get("split")\n    if not isinstance(split, dict):\n        raise ValueError("checkpoint split is missing or incompatible")\n    if (split.get("seed"), split.get("block_size"), split.get("guard")) != (FIXED_SEED, FIXED_BLOCK_SIZE, FIXED_GUARD):\n        raise ValueError("checkpoint split is not the fixed Team 23 scalar split")\n    labels, expected = split_labels(\n        total_length,\n        seed=FIXED_SEED,\n        block_size=FIXED_BLOCK_SIZE,\n        guard=FIXED_GUARD,\n    )\n    expected["labels_sha256"] = hashlib.sha256(labels.tobytes()).hexdigest()\n    if split != expected:\n        raise ValueError("checkpoint split hash or specification does not match")\n    return expected\n\n\ndef _load_validated_checkpoint(checkpoint_path: Path, data_dir: Path, total_length: int) -> tuple[dict[str, Any], ScalarResidualMLP, dict[str, str], dict[str, Any]]:\n    try:\n        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    except Exception as exc:\n        raise ValueError("unable to read scalar checkpoint") from exc\n    if not isinstance(payload, dict) or payload.get("schema_version") != SCHEMA_VERSION:\n        raise ValueError("checkpoint schema is incomplete or unsupported")\n    config = _validate_config(payload.get("config"))\n    input_hashes = _validate_hashes(payload, data_dir)\n    clean = np.load(data_dir / "train_weights_clean.npy", mmap_mode="r", allow_pickle=False)\n    noisy = np.load(data_dir / "train_weights_noisy.npy", mmap_mode="r", allow_pickle=False)\n    if clean.shape != (total_length,) or noisy.shape != (total_length,) or clean.dtype != np.float32 or noisy.dtype != np.float32:\n        raise ValueError("training arrays are incompatible with the checkpoint")\n    split = _validate_split(payload, total_length)\n    expected = {"config": config, "input_sha256": input_hashes, "split": split}\n    _validate_checkpoint(payload, expected)\n    if not isinstance(payload["optimizer"], dict) or not payload["optimizer"].get("state") or not payload["optimizer"].get("param_groups"):\n        raise ValueError("checkpoint optimizer state is missing or incomplete")\n    if not isinstance(payload["scheduler"], dict) or type(payload["scheduler"].get("last_epoch")) is not int or type(payload["scheduler"].get("_step_count")) is not int or payload["scheduler"]["last_epoch"] < -1 or payload["scheduler"]["_step_count"] < 0:\n        raise ValueError("checkpoint scheduler state is missing or incomplete")\n    if not (0 <= payload["epoch"] <= config["epochs"] and payload["step"] >= 0):\n        raise ValueError("checkpoint progress counters are invalid")\n    rng = payload["rng"]\n    if not isinstance(rng, dict) or set(rng) - {"python", "numpy", "torch", "mps"} or not {"python", "numpy", "torch"}.issubset(rng):\n        raise ValueError("checkpoint RNG state is missing or incomplete")\n    if not isinstance(rng["python"], tuple) or not isinstance(rng["numpy"], tuple) or not isinstance(rng["torch"], torch.Tensor):\n        raise ValueError("checkpoint RNG state is missing or incomplete")\n    for key in ("model", "optimizer", "scheduler", "rng"):\n        _finite_tree(payload[key], key)\n    model = ScalarResidualMLP(config["hidden"])\n    try:\n        model.load_state_dict(payload["model"], strict=True)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"]))\n        optimizer.load_state_dict(payload["optimizer"])\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(config["epochs"]))\n        scheduler.load_state_dict(payload["scheduler"])\n    except Exception as exc:\n        raise ValueError("checkpoint model state is incompatible") from exc\n    model.eval()\n    return payload, model, input_hashes, split\n\n\ndef _atomic_save_npy(path: Path, values: np.ndarray) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp", delete=False) as handle:\n        temporary = Path(handle.name)\n    try:\n        np.save(temporary, values, allow_pickle=False)\n        generated = temporary.with_suffix(temporary.suffix + ".npy")\n        os.replace(generated, path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n        generated = temporary.with_suffix(temporary.suffix + ".npy")\n        if generated.exists():\n            generated.unlink()\n\n\ndef _write_csv(path: Path, values: np.ndarray) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp", mode="w", newline="", encoding="utf-8", delete=False) as handle:\n        temporary = Path(handle.name)\n        writer = csv.writer(handle, lineterminator="\\n")\n        writer.writerow(("id", "weight"))\n        for index, value in enumerate(values):\n            writer.writerow((index, repr(float(value))))\n    os.replace(temporary, path)\n\n\ndef _verify_outputs(npy_path: Path, csv_path: Path, values: np.ndarray) -> None:\n    saved = np.load(npy_path, allow_pickle=False)\n    if saved.dtype != np.float32 or saved.shape != values.shape or not np.array_equal(saved, values):\n        raise ValueError("saved NPY does not match the restored vector")\n    with csv_path.open("r", encoding="utf-8", newline="") as handle:\n        reader = csv.reader(handle)\n        if next(reader, None) != ["id", "weight"]:\n            raise ValueError("CSV header must be exactly id,weight")\n        count = 0\n        for row in reader:\n            if count >= values.size:\n                raise ValueError("CSV contains more rows than the restored vector")\n            if len(row) != 2 or row[0] != str(count):\n                raise ValueError("CSV IDs are missing, reordered, or duplicated")\n            try:\n                parsed = np.float32(row[1])\n            except (TypeError, ValueError, OverflowError) as exc:\n                raise ValueError("CSV contains an invalid weight") from exc\n            if not np.isfinite(parsed) or parsed != values[count]:\n                raise ValueError("CSV weights do not match the restored NPY")\n            count += 1\n        if count != values.size:\n            raise ValueError("CSV row count does not match the restored vector")\n\n\ndef export(checkpoint: Path, data_dir: Path, npy_output: Path, csv_output: Path, *, chunk_size: int = 65_536, expected_count: int = EXPECTED_TEST_COUNT) -> dict[str, Any]:\n    if chunk_size <= 0 or expected_count <= 0:\n        raise ValueError("chunk_size and expected_count must be positive")\n    protected = {checkpoint.resolve(), data_dir.resolve() / "manifest.json", data_dir.resolve() / "test_weights_noisy.npy", data_dir.resolve() / "train_weights_clean.npy", data_dir.resolve() / "train_weights_noisy.npy"}\n    output_paths = {npy_output.resolve(), csv_output.resolve(), npy_output.with_suffix(".json").resolve()}\n    if len(output_paths) != 3 or protected & output_paths or any(path.is_relative_to(data_dir.resolve()) for path in output_paths):\n        raise ValueError("output paths must be distinct and cannot overwrite inputs or checkpoint")\n    test_path = data_dir / "test_weights_noisy.npy"\n    if not test_path.is_file():\n        raise ValueError("test noisy weights are missing")\n    test = np.load(test_path, mmap_mode="r", allow_pickle=False)\n    if test.ndim != 1 or test.dtype != np.float32 or test.size != expected_count:\n        raise ValueError("test noisy vector has invalid dtype, shape, or exact count")\n    if not np.isfinite(test).all():\n        raise ValueError("test noisy vector contains non-finite values")\n    manifest_path = data_dir / "manifest.json"\n    if not manifest_path.is_file():\n        raise ValueError("Team 23 manifest is missing")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    test_hash = _hash(test_path)\n    if manifest.get("files", {}).get(test_path.name, {}).get("sha256") != test_hash:\n        raise ValueError("test weight hash differs from the preflight manifest")\n    clean_header = np.load(data_dir / "train_weights_clean.npy", mmap_mode="r", allow_pickle=False)\n    payload, model, input_hashes, split = _load_validated_checkpoint(checkpoint, data_dir, int(clean_header.size))\n    restored = np.empty(expected_count, dtype=np.float32)\n    with torch.no_grad():\n        for start in range(0, expected_count, chunk_size):\n            stop = min(start + chunk_size, expected_count)\n            batch = torch.from_numpy(np.array(test[start:stop], copy=True)).to(dtype=torch.float32)\n            prediction = model(batch).numpy()\n            if prediction.shape != (stop - start,) or not np.isfinite(prediction).all():\n                raise ValueError("checkpoint inference produced an invalid chunk")\n            restored[start:stop] = prediction.astype(np.float32, copy=False)\n    if restored.size != expected_count or not np.isfinite(restored).all():\n        raise ValueError("restored vector has invalid values or exact count")\n    _atomic_save_npy(npy_output, restored)\n    _write_csv(csv_output, restored)\n    _verify_outputs(npy_output, csv_output, restored)\n    metadata = {\n        "schema_version": 2,\n        "method": "schema2_scalar_residual_mlp",\n        "shape": [int(restored.size)],\n        "dtype": "float32",\n        "checkpoint_sha256": _hash(checkpoint),\n        "input_sha256": input_hashes,\n        "split_sha256": split["labels_sha256"],\n        "split_spec_sha256": split["spec_sha256"],\n        "test_noisy_sha256": test_hash,\n        "restored_npy_sha256": _hash(npy_output),\n        "submission_csv_sha256": _hash(csv_output),\n        "submission_rows": int(restored.size),\n        "audit_metrics": None,\n        "config": payload["config"],\n    }\n    metadata_path = npy_output.with_suffix(".json")\n    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return metadata\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint", type=Path, default=DEFAULT_CHECKPOINT)\n    parser.add_argument("--data-dir", type=Path, default=DEFAULT_DATA)\n    parser.add_argument("--npy-output", type=Path, default=DEFAULT_NPY)\n    parser.add_argument("--csv-output", type=Path, default=DEFAULT_CSV)\n    parser.add_argument("--chunk-size", type=int, default=65_536)\n    args = parser.parse_args()\n    print(json.dumps(export(args.checkpoint, args.data_dir, args.npy_output, args.csv_output, chunk_size=args.chunk_size), sort_keys=True))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/preflight_team23.py': '#!/usr/bin/env python3\n"""Read-only Team 23 input preflight and manifest writer."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport struct\nimport sys\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nDEFAULT_DATA_DIR = REPO_ROOT / "data" / "23_Team_Toxic"\nMANIFEST_NAME = "manifest.json"\nREQUIRED_FILES = (\n    "boilerplate.ipynb",\n    "test_weights_noisy.npy",\n    "train_weights_clean.npy",\n    "train_weights_noisy.npy",\n    "test_queries.csv",\n    "train_parallel_corpus.csv",\n)\nEXPECTED_ARRAYS = {\n    "train_weights_clean.npy": 5_923_260,\n    "train_weights_noisy.npy": 5_923_260,\n    "test_weights_noisy.npy": 2_961_630,\n}\nEXPECTED_ROWS = {"test_queries.csv": 200, "train_parallel_corpus.csv": 300}\nCHUNK_ELEMENTS = 1_048_576\nHASH_CHUNK_BYTES = 1024 * 1024\n\n\nclass PreflightError(Exception):\n    """A user-correctable input or validation failure."""\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    try:\n        with path.open("rb") as handle:\n            while chunk := handle.read(HASH_CHUNK_BYTES):\n                digest.update(chunk)\n    except OSError as exc:\n        raise PreflightError(f"cannot hash {path.name}: {exc}") from exc\n    return digest.hexdigest()\n\n\ndef npy_header(path: Path) -> tuple[dict[str, Any], int, str]:\n    """Read only the NPY header; never enables pickle or memory mapping."""\n    try:\n        with path.open("rb") as handle:\n            if handle.read(6) != b"\\x93NUMPY":\n                raise PreflightError(f"{path.name}: invalid NPY magic")\n            version_bytes = handle.read(2)\n            if len(version_bytes) != 2:\n                raise PreflightError(f"{path.name}: truncated NPY version")\n            major, minor = version_bytes\n            if (major, minor) == (1, 0):\n                length_bytes = handle.read(2)\n                length_format = "<H"\n            elif (major, minor) in {(2, 0), (3, 0)}:\n                length_bytes = handle.read(4)\n                length_format = "<I"\n            else:\n                raise PreflightError(f"{path.name}: unsupported NPY version {(major, minor)}")\n            if len(length_bytes) != struct.calcsize(length_format):\n                raise PreflightError(f"{path.name}: truncated NPY header length")\n            header_length = struct.unpack(length_format, length_bytes)[0]\n            if header_length > 10_000:\n                raise PreflightError(f"{path.name}: NPY header exceeds 10000 bytes")\n            raw_header = handle.read(header_length)\n            if len(raw_header) != header_length:\n                raise PreflightError(f"{path.name}: truncated NPY header")\n            try:\n                metadata = ast.literal_eval(raw_header.decode("utf-8"))\n            except (SyntaxError, UnicodeDecodeError, ValueError) as exc:\n                raise PreflightError(f"{path.name}: malformed NPY header: {exc}") from exc\n            if not isinstance(metadata, dict):\n                raise PreflightError(f"{path.name}: NPY header is not a dictionary")\n            offset = handle.tell()\n    except OSError as exc:\n        raise PreflightError(f"{path.name}: invalid NPY header: {exc}") from exc\n    return metadata, offset, f"{major}.{minor}"\n\n\ndef inspect_npy(path: Path, expected_length: int) -> dict[str, Any]:\n    metadata, offset, version = npy_header(path)\n    descr = metadata.get("descr")\n    shape = metadata.get("shape")\n    if descr != "<f4":\n        raise PreflightError(f"{path.name}: expected little-endian float32, got {descr!r}")\n    if metadata.get("fortran_order") not in (False,):\n        raise PreflightError(f"{path.name}: expected a C-order 1D array")\n    if not isinstance(shape, tuple) or len(shape) != 1 or shape[0] != expected_length:\n        raise PreflightError(f"{path.name}: expected shape ({expected_length},), got {shape!r}")\n\n    finite_checked = 0\n    try:\n        with path.open("rb") as handle:\n            handle.seek(offset)\n            while finite_checked < expected_length:\n                count = min(CHUNK_ELEMENTS, expected_length - finite_checked)\n                raw = handle.read(count * 4)\n                if len(raw) != count * 4:\n                    raise PreflightError(\n                        f"{path.name}: truncated payload at element {finite_checked} "\n                        f"(expected {expected_length})"\n                    )\n                values = struct.unpack(f"<{count}f", raw)\n                if not all(math.isfinite(value) for value in values):\n                    raise PreflightError(f"{path.name}: non-finite value found near element {finite_checked}")\n                finite_checked += count\n    except OSError as exc:\n        raise PreflightError(f"{path.name}: cannot read payload: {exc}") from exc\n    return {\n        "kind": "npy",\n        "sha256": sha256_file(path),\n        "dtype": "float32",\n        "shape": [expected_length],\n        "npy_version": version,\n        "finite_values_checked": finite_checked,\n    }\n\n\ndef inspect_csv(path: Path, expected_rows: int) -> dict[str, Any]:\n    try:\n        with path.open("r", encoding="utf-8-sig", newline="") as handle:\n            reader = csv.reader(handle)\n            try:\n                header = next(reader)\n            except StopIteration as exc:\n                raise PreflightError(f"{path.name}: empty CSV") from exc\n            if not header or any(not column.strip() for column in header):\n                raise PreflightError(f"{path.name}: header contains an empty column name")\n            if len(set(header)) != len(header):\n                raise PreflightError(f"{path.name}: duplicate header names")\n            try:\n                id_index = next(index for index, name in enumerate(header) if name.strip().lower() == "id")\n            except StopIteration as exc:\n                raise PreflightError(f"{path.name}: CSV header must contain an id column") from exc\n            ids: set[str] = set()\n            rows = 0\n            for row_number, row in enumerate(reader, start=2):\n                if len(row) != len(header):\n                    raise PreflightError(f"{path.name}: row {row_number} has {len(row)} fields, expected {len(header)}")\n                identifier = row[id_index].strip()\n                if not identifier:\n                    raise PreflightError(f"{path.name}: empty id at row {row_number}")\n                if identifier in ids:\n                    raise PreflightError(f"{path.name}: duplicate id {identifier!r} at row {row_number}")\n                ids.add(identifier)\n                rows += 1\n    except UnicodeDecodeError as exc:\n        raise PreflightError(f"{path.name}: invalid UTF-8 CSV: {exc}") from exc\n    except OSError as exc:\n        raise PreflightError(f"{path.name}: cannot read CSV: {exc}") from exc\n    if rows != expected_rows:\n        raise PreflightError(f"{path.name}: expected {expected_rows} data rows, found {rows}")\n    return {\n        "kind": "csv",\n        "sha256": sha256_file(path),\n        "header": header,\n        "row_count": rows,\n        "unique_ids": len(ids),\n    }\n\n\ndef inspect_notebook(path: Path) -> dict[str, Any]:\n    try:\n        with path.open("r", encoding="utf-8") as handle:\n            notebook = json.load(handle)\n    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise PreflightError(f"{path.name}: invalid notebook JSON: {exc}") from exc\n    cells = notebook.get("cells") if isinstance(notebook, dict) else None\n    if not isinstance(cells, list):\n        raise PreflightError(f"{path.name}: notebook JSON has no cells list")\n    return {"kind": "notebook", "sha256": sha256_file(path), "cell_count": len(cells)}\n\n\ndef atomic_write_json(path: Path, value: dict[str, Any]) -> None:\n    payload = json.dumps(value, indent=2, sort_keys=True) + "\\n"\n    fd, temporary = tempfile.mkstemp(prefix=".manifest.", suffix=".tmp", dir=str(path.parent), text=True)\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as handle:\n            handle.write(payload)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        try:\n            os.unlink(temporary)\n        except FileNotFoundError:\n            pass\n\n\ndef run(data_dir: Path) -> int:\n    manifest_path = data_dir / MANIFEST_NAME\n    # A prior success must not be mistaken for the result of a failed current run.\n    try:\n        if manifest_path.exists() or manifest_path.is_symlink():\n            manifest_path.unlink()\n    except OSError as exc:\n        print(f"preflight error: cannot remove old manifest: {exc}", file=sys.stderr)\n        return 1\n\n    try:\n        if not data_dir.is_dir():\n            raise PreflightError(f"data directory does not exist: {data_dir}")\n        present = {entry.name for entry in data_dir.iterdir()}\n        required = set(REQUIRED_FILES)\n        missing = sorted(required - present)\n        unexpected = sorted(present - required - {MANIFEST_NAME})\n        if missing:\n            raise PreflightError("missing required file(s): " + ", ".join(missing))\n        if unexpected:\n            raise PreflightError("unexpected input file(s): " + ", ".join(unexpected))\n\n        files: dict[str, Any] = {}\n        for name in REQUIRED_FILES:\n            path = data_dir / name\n            if name in EXPECTED_ARRAYS:\n                files[name] = inspect_npy(path, EXPECTED_ARRAYS[name])\n            elif name in EXPECTED_ROWS:\n                files[name] = inspect_csv(path, EXPECTED_ROWS[name])\n            else:\n                files[name] = inspect_notebook(path)\n        manifest = {\n            "schema_version": 1,\n            "team": 23,\n            "data_directory": str(data_dir.resolve()),\n            "required_files": list(REQUIRED_FILES),\n            "files": files,\n        }\n        atomic_write_json(manifest_path, manifest)\n    except (OSError, PreflightError) as exc:\n        print(f"preflight error: {exc}", file=sys.stderr)\n        return 1\n    print(f"preflight ok: wrote {manifest_path}")\n    return 0\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data-dir", type=Path, default=DEFAULT_DATA_DIR, help="Team 23 input directory")\n    args = parser.parse_args(argv)\n    return run(args.data_dir)\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'scripts/make_translation_split.py': '#!/usr/bin/env python3\n"""Create a deterministic, source-only grouped split for the Team 23 corpus."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport tempfile\nimport unicodedata\nfrom pathlib import Path\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nDEFAULT_CORPUS = REPO_ROOT / "data" / "23_Team_Toxic" / "train_parallel_corpus.csv"\nDEFAULT_OUTPUT = REPO_ROOT / "data" / "23_Team_Toxic" / "translation_split.json"\nRULE_VERSION = "source-group-v1"\nDEFAULT_SEED = 2301\nTARGETS_300 = {"fitting": 210, "development": 45, "audit": 45}\n_DIGITS = re.compile(r"\\d+")\n_HEX_TOKEN = re.compile(r"(?i)(?=[a-f0-9]{8,}\\b)[a-f0-9]+\\b")\n_WHITESPACE = re.compile(r"\\s+")\n\n\nclass SplitError(ValueError):\n    """Raised when the corpus cannot safely receive a split."""\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef normalize_source(source: str) -> str:\n    """Canonical source form used for conservative grouping."""\n    return _WHITESPACE.sub(" ", unicodedata.normalize("NFKC", source)).strip().casefold()\n\n\ndef template_source(source: str) -> str:\n    """Mask only digit runs and long hex-like tokens after canonicalization."""\n    normalized = normalize_source(source)\n    masked = _HEX_TOKEN.sub("<HEX>", normalized)\n    return _DIGITS.sub("<DIGITS>", masked)\n\n\ndef _source_keys(source: str) -> tuple[str, str, str]:\n    return (f"exact:{source}", f"normalized:{normalize_source(source)}", f"template:{template_source(source)}")\n\n\ndef read_rows(path: Path, expected_rows: int | None = 300) -> list[dict[str, str]]:\n    with path.open("r", encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames is None or "id" not in reader.fieldnames or "asdfghjkl" not in reader.fieldnames:\n            raise SplitError("corpus must contain id and asdfghjkl columns")\n        rows = list(reader)\n    if expected_rows is not None and len(rows) != expected_rows:\n        raise SplitError(f"expected {expected_rows} rows, found {len(rows)}")\n    ids = [row.get("id", "").strip() for row in rows]\n    if any(not identifier for identifier in ids):\n        raise SplitError("corpus contains an empty id")\n    if len(set(ids)) != len(ids):\n        raise SplitError("corpus contains duplicate ids")\n    if any(not row.get("asdfghjkl", "").strip() for row in rows):\n        raise SplitError("corpus contains an empty source")\n    return rows\n\n\ndef _groups(rows: list[dict[str, str]]) -> list[list[int]]:\n    parent = list(range(len(rows)))\n\n    def find(index: int) -> int:\n        while parent[index] != index:\n            parent[index] = parent[parent[index]]\n            index = parent[index]\n        return index\n\n    def union(left: int, right: int) -> None:\n        left, right = find(left), find(right)\n        if left != right:\n            parent[right] = left\n\n    first_by_key: dict[str, int] = {}\n    for index, row in enumerate(rows):\n        for key in _source_keys(row["asdfghjkl"]):\n            previous = first_by_key.setdefault(key, index)\n            union(index, previous)\n    grouped: dict[int, list[int]] = {}\n    for index in range(len(rows)):\n        grouped.setdefault(find(index), []).append(index)\n    return sorted((sorted(indices) for indices in grouped.values()), key=lambda x: x[0])\n\n\ndef _scaled_targets(total: int) -> dict[str, int]:\n    if total == 300:\n        return dict(TARGETS_300)\n    development = round(total * 0.15)\n    audit = round(total * 0.15)\n    return {"fitting": total - development - audit, "development": development, "audit": audit}\n\n\ndef assign_groups(rows: list[dict[str, str]], seed: int = DEFAULT_SEED, targets: dict[str, int] | None = None) -> dict[str, list[str]]:\n    """Assign complete source groups with a stable seeded ordering."""\n    targets = dict(targets or _scaled_targets(len(rows)))\n    if set(targets) != set(TARGETS_300) or sum(targets.values()) != len(rows):\n        raise SplitError("split targets must cover every row exactly once")\n    names = tuple(TARGETS_300)\n    groups = _groups(rows)\n    order = sorted(\n        groups,\n        key=lambda group: hashlib.sha256(\n            f"{seed}\\0{\',\'.join(rows[i][\'id\'] for i in group)}".encode("utf-8")\n        ).hexdigest(),\n    )\n    order.sort(key=len, reverse=True)\n    counts = {name: 0 for name in names}\n    assigned = {name: [] for name in names}\n    for group in order:\n        size = len(group)\n        fitting = [name for name in names if counts[name] + size <= targets[name]]\n        if not fitting:\n            raise SplitError("source group cannot fit any remaining split target")\n        candidates = fitting\n        chosen = min(\n            candidates,\n            key=lambda name: (\n                counts[name] / targets[name] if targets[name] else float("inf"),\n                hashlib.sha256(f"{seed}\\0{name}".encode("utf-8")).hexdigest(),\n            ),\n        )\n        counts[chosen] += size\n        assigned[chosen].extend(rows[i]["id"] for i in group)\n    return {name: sorted(identifiers) for name, identifiers in assigned.items()}\n\n\ndef make_split(corpus: Path, seed: int = DEFAULT_SEED, expected_rows: int | None = 300) -> dict:\n    if corpus.resolve() == DEFAULT_CORPUS.resolve():\n        manifest_path = corpus.parent / "manifest.json"\n        if not manifest_path.is_file():\n            raise SplitError("Team 23 input manifest is missing; run preflight first")\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        expected_hash = manifest["files"][corpus.name]["sha256"]\n        if sha256_file(corpus) != expected_hash:\n            raise SplitError("Team 23 corpus hash differs from preflight manifest")\n    rows = read_rows(corpus, expected_rows=expected_rows)\n    assignments = assign_groups(rows, seed=seed)\n    all_ids = sorted(row["id"].strip() for row in rows)\n    split_ids = sorted(identifier for identifiers in assignments.values() for identifier in identifiers)\n    if split_ids != all_ids or any(not identifiers for identifiers in assignments.values()):\n        raise SplitError("split validation failed: IDs are not exhaustive or a split is empty")\n    return {\n        "schema_version": 1,\n        "rule_version": RULE_VERSION,\n        "seed": seed,\n        "source_corpus": corpus.name,\n        "source_corpus_sha256": sha256_file(corpus),\n        "row_count": len(rows),\n        "group_count": len(_groups(rows)),\n        "counts": {name: len(identifiers) for name, identifiers in assignments.items()},\n        "splits": assignments,\n    }\n\n\ndef write_json(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False) as handle:\n        temporary = Path(handle.name)\n        json.dump(payload, handle, indent=2, sort_keys=True)\n        handle.write("\\n")\n    temporary.replace(path)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--corpus", type=Path, default=DEFAULT_CORPUS)\n    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)\n    parser.add_argument("--seed", type=int, default=DEFAULT_SEED)\n    parser.add_argument("--expected-rows", type=int, default=300, help="row-count gate (use only for synthetic tests)")\n    args = parser.parse_args()\n    try:\n        payload = make_split(args.corpus, seed=args.seed, expected_rows=args.expected_rows)\n        write_json(args.output, payload)\n    except (OSError, SplitError) as exc:\n        parser.error(str(exc))\n    print(f"wrote {args.output}: " + ", ".join(f"{k}={v}" for k, v in payload["counts"].items()))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'scripts/validate_translation_split.py': '"""Fail closed unless the locally frozen Team 23 split is exactly reproduced."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom scripts.make_translation_split import make_split, sha256_file\n\nAPPROVED_SPLIT_SHA256 = "65fddacd277c3bdb87a7f3bb879dda220957ae5f8cdef966f5f3b1ee1abf92bf"\nAPPROVED_COUNTS = {"fitting": 210, "development": 45, "audit": 45}\n\n\ndef validate_frozen_split(corpus: Path, split_path: Path) -> dict:\n    if sha256_file(split_path) != APPROVED_SPLIT_SHA256:\n        raise ValueError("translation split hash differs from the approved Team 23 split")\n    payload = json.loads(split_path.read_text(encoding="utf-8"))\n    if payload.get("schema_version") != 1 or payload.get("rule_version") != "source-group-v1" or payload.get("counts") != APPROVED_COUNTS:\n        raise ValueError("translation split schema, rule, or counts differ from the approved plan")\n    expected = make_split(corpus, seed=2301, expected_rows=300)\n    if payload != expected:\n        raise ValueError("translation split does not match source-only grouped reconstruction")\n    return payload\n', 'scripts/team23_tokenizer.py': '"""Load the supplied embedded vocabularies while preserving their token IDs."""\n\nfrom __future__ import annotations\n\nimport ast\nimport base64\nimport json\nimport re\nimport zlib\nfrom pathlib import Path\n\nEXPECTED_SPECIAL = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}\nEMBEDDED_NAMES = {\n    "source": "_EMBEDDED_VOCAB_ASDF_B64",\n    "target": "_EMBEDDED_VOCAB_ENG_B64",\n}\n\n\ndef load_embedded_vocab(notebook_path: Path, language: str) -> dict[str, int]:\n    """Extract one vocabulary from the actual assigned notebook, not a rebuilt corpus."""\n    assignment = EMBEDDED_NAMES[language]\n    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))\n    for cell in notebook["cells"]:\n        source = "".join(cell.get("source", []))\n        if assignment not in source:\n            continue\n        for node in ast.parse(source).body:\n            if isinstance(node, ast.Assign) and any(\n                isinstance(target, ast.Name) and target.id == assignment for target in node.targets\n            ):\n                encoded = ast.literal_eval(node.value)\n                vocab = json.loads(zlib.decompress(base64.b64decode(encoded)).decode("utf-8"))\n                if not isinstance(vocab, dict) or any(\n                    not isinstance(token, str) or not isinstance(index, int) for token, index in vocab.items()\n                ):\n                    raise ValueError(f"{language} vocabulary has invalid entries")\n                if {name: vocab.get(name) for name in EXPECTED_SPECIAL} != EXPECTED_SPECIAL:\n                    raise ValueError(f"{language} special-token IDs differ from supplied contract")\n                indices = sorted(vocab.values())\n                if indices != list(range(len(vocab))):\n                    raise ValueError(f"{language} vocabulary IDs are not unique and contiguous")\n                return vocab\n    raise ValueError(f"{language} embedded vocabulary not found")\n\n\nclass Team23Tokenizer:\n    def __init__(self, vocab: dict[str, int], *, is_source: bool):\n        self.word2idx = dict(vocab)\n        self.idx2word = {index: word for word, index in vocab.items()}\n        self.is_source = is_source\n        self.pad_id = vocab["<pad>"]\n        self.bos_id = vocab["<bos>"]\n        self.eos_id = vocab["<eos>"]\n        self.unk_id = vocab["<unk>"]\n\n    @property\n    def vocab_size(self) -> int:\n        return len(self.word2idx)\n\n    def tokenize(self, text: str) -> list[str]:\n        if self.is_source:\n            return [token.strip() for token in str(text).strip().split() if token.strip()]\n        tokens = []\n        for token in str(text).strip().split():\n            cleaned = re.sub(r"[^\\w\'-]", "", token, flags=re.UNICODE).lower()\n            if cleaned:\n                tokens.append(cleaned)\n        return tokens\n\n    def encode(self, text: str, *, add_special_tokens: bool = True) -> list[int]:\n        ids = [self.word2idx.get(token, self.unk_id) for token in self.tokenize(text)]\n        return [self.bos_id, *ids, self.eos_id] if add_special_tokens else ids\n\n    def decode(self, ids: list[int], *, skip_special_tokens: bool = True) -> str:\n        words = []\n        for index in ids:\n            if skip_special_tokens and index == self.eos_id:\n                break\n            if skip_special_tokens and index in {self.pad_id, self.bos_id, self.unk_id}:\n                continue\n            word = self.idx2word.get(index)\n            if word is not None:\n                words.append(word)\n        return " ".join(words)\n\n\ndef load_tokenizers(notebook_path: Path) -> tuple[Team23Tokenizer, Team23Tokenizer]:\n    return (\n        Team23Tokenizer(load_embedded_vocab(notebook_path, "source"), is_source=True),\n        Team23Tokenizer(load_embedded_vocab(notebook_path, "target"), is_source=False),\n    )\n', 'scripts/translation_checkpoint.py': '"""Fail-closed checks for locally produced translation trainer checkpoints."""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Any\n\nimport torch\n\n\ndef _check_finite(value: Any, path: str) -> None:\n    if isinstance(value, torch.Tensor):\n        if not bool(torch.isfinite(value).all()):\n            raise ValueError(f"non-finite checkpoint tensor: {path}")\n    elif isinstance(value, float):\n        if not math.isfinite(value):\n            raise ValueError(f"non-finite checkpoint value: {path}")\n    elif isinstance(value, dict):\n        for key, item in value.items():\n            _check_finite(item, f"{path}.{key}")\n    elif isinstance(value, (list, tuple)):\n        for index, item in enumerate(value):\n            _check_finite(item, f"{path}[{index}]")\n\n\ndef validate_checkpoint(\n    payload: Any,\n    *,\n    input_hashes: dict[str, str],\n    split_ids: dict[str, list[str]],\n    parameter_count: int,\n    special_ids: dict[str, int],\n) -> None:\n    """Validate inference provenance and the complete recovery envelope.\n\n    This does not deserialize untrusted files or replace strict model loading.\n    It deliberately permits legacy v1 checkpoints without newer lineage fields.\n    """\n    required = {"contract", "model", "optimizer", "scheduler", "epoch", "step",\n                "rng", "config", "input_hashes", "split_ids"}\n    if not isinstance(payload, dict) or not required.issubset(payload):\n        raise ValueError("translation checkpoint is missing required contract fields")\n    if payload["contract"] != "translation-trainer-v1":\n        raise ValueError("translation checkpoint contract is unsupported")\n    for key in ("epoch", "step"):\n        if type(payload[key]) is not int or payload[key] < 0:\n            raise ValueError(f"checkpoint {key} must be a nonnegative integer")\n    if payload["input_hashes"] != input_hashes or payload["split_ids"] != split_ids:\n        raise ValueError("translation checkpoint inputs or split IDs do not match")\n    config = payload["config"]\n    if (not isinstance(config, dict)\n            or config.get("trainer") != "translation-trainer-v1"\n            or config.get("architecture") != "supplied_TranslationTransformer_v1"\n            or config.get("parameter_count") != parameter_count\n            or config.get("smoke") is not False\n            or config.get("special_ids") != special_ids):\n        raise ValueError("translation checkpoint configuration is incompatible")\n    model = payload["model"]\n    if (not isinstance(model, dict) or not model\n            or any(not isinstance(v, torch.Tensor) for v in model.values())):\n        raise ValueError("translation checkpoint model state must contain tensors")\n    optimizer, scheduler = payload["optimizer"], payload["scheduler"]\n    if (not isinstance(optimizer, dict) or not {"state", "param_groups"}.issubset(optimizer)\n            or not isinstance(optimizer["state"], dict)\n            or not isinstance(optimizer["param_groups"], list)\n            or not optimizer["param_groups"]\n            or not isinstance(scheduler, dict) or "last_epoch" not in scheduler):\n        raise ValueError("translation checkpoint optimizer/scheduler state is incomplete")\n    rng = payload["rng"]\n    if (not isinstance(rng, dict) or not {"python", "numpy", "torch"}.issubset(rng)\n            or not isinstance(rng["python"], tuple)\n            or not isinstance(rng["numpy"], tuple)\n            or not isinstance(rng["torch"], torch.Tensor)\n            or rng["torch"].dtype != torch.uint8 or rng["torch"].ndim != 1):\n        raise ValueError("translation checkpoint RNG state is incomplete")\n    for key in ("model", "optimizer", "scheduler"):\n        _check_finite(payload[key], key)\n', 'scripts/translation_noisy_baseline.py': '#!/usr/bin/env python3\n"""Evaluate the supplied noisy translation weights on Team 23 development only.\n\nThis is an inference-only baseline.  It deliberately does not read test queries,\nfit on any rows, or alter the supplied weights.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport platform\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nimport sacrebleu\nimport torch\nfrom sacrebleu.metrics import BLEU, CHRF\n\nfrom scripts.team23_tokenizer import Team23Tokenizer, load_tokenizers\nfrom scripts.validate_translation_split import validate_frozen_split\nfrom scripts.translation_checkpoint import validate_checkpoint\n\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / "data" / "23_Team_Toxic"\nDEFAULT_NOTEBOOK = DATA / "boilerplate.ipynb"\nDEFAULT_CORPUS = DATA / "train_parallel_corpus.csv"\nDEFAULT_WEIGHTS = DATA / "test_weights_noisy.npy"\nDEFAULT_SPLIT = DATA / "translation_split.json"\nDEFAULT_JSON = ROOT / "results" / "baselines" / "translation_noisy_dev.json"\nDEFAULT_CSV = ROOT / "results" / "baselines" / "translation_noisy_dev.csv"\nEXPECTED_PARAMETER_COUNT = 2_961_630\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass PositionalEncoding(torch.nn.Module):\n    def __init__(self, d_model: int, max_len: int = 256):\n        super().__init__()\n        pe = torch.zeros(max_len, d_model)\n        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)\n        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))\n        pe[:, 0::2] = torch.sin(position * div_term)\n        pe[:, 1::2] = torch.cos(position * div_term)\n        self.register_buffer("pe", pe.unsqueeze(0))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return x + self.pe[:, : x.size(1), :]\n\n\nclass TranslationTransformer(torch.nn.Module):\n    """The supplied notebook architecture, with its parameter order unchanged."""\n\n    def __init__(\n        self,\n        src_vocab_size: int = 5947,\n        tgt_vocab_size: int = 5982,\n        d_model: int = 128,\n        nhead: int = 4,\n        num_encoder_layers: int = 2,\n        num_decoder_layers: int = 2,\n        dim_feedforward: int = 256,\n        dropout: float = 0.0,\n    ):\n        super().__init__()\n        self.d_model = d_model\n        self.src_embedding = torch.nn.Embedding(src_vocab_size, d_model)\n        self.tgt_embedding = torch.nn.Embedding(tgt_vocab_size, d_model)\n        self.pos_encoder = PositionalEncoding(d_model)\n        self.transformer = torch.nn.Transformer(\n            d_model=d_model,\n            nhead=nhead,\n            num_encoder_layers=num_encoder_layers,\n            num_decoder_layers=num_decoder_layers,\n            dim_feedforward=dim_feedforward,\n            dropout=dropout,\n            batch_first=True,\n        )\n        self.fc_out = torch.nn.Linear(d_model, tgt_vocab_size)\n\n    def generate_square_subsequent_mask(self, sz: int, device: torch.device) -> torch.Tensor:\n        mask = (torch.triu(torch.ones(sz, sz, device=device)) == 1).transpose(0, 1)\n        return mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, float(0.0))\n\n    def forward(\n        self,\n        src: torch.Tensor,\n        tgt: torch.Tensor,\n        src_pad_idx: int = 0,\n        tgt_pad_idx: int = 0,\n    ) -> torch.Tensor:\n        device = src.device\n        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1), device)\n        src_padding_mask = src == src_pad_idx\n        tgt_padding_mask = tgt == tgt_pad_idx\n        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))\n        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))\n        out = self.transformer(\n            src_emb,\n            tgt_emb,\n            tgt_mask=tgt_mask,\n            src_key_padding_mask=src_padding_mask,\n            tgt_key_padding_mask=tgt_padding_mask,\n        )\n        return self.fc_out(out)\n\n    def load_flattened_weights(self, flat_weights: np.ndarray | torch.Tensor) -> None:\n        values = torch.as_tensor(flat_weights, dtype=torch.float32).reshape(-1)\n        parameters = list(self.named_parameters())\n        expected = sum(parameter.numel() for _, parameter in parameters)\n        if values.numel() != expected:\n            raise ValueError(f"weight length {values.numel()} does not equal model parameter count {expected}")\n        offset = 0\n        with torch.no_grad():\n            for _, parameter in parameters:\n                count = parameter.numel()\n                parameter.copy_(values[offset : offset + count].view_as(parameter))\n                offset += count\n\n\ndef prepare_model(weights: np.ndarray, device: torch.device) -> TranslationTransformer:\n    model = TranslationTransformer()\n    if sum(parameter.numel() for parameter in model.parameters()) != EXPECTED_PARAMETER_COUNT:\n        raise RuntimeError("fixed Transformer parameter count changed")\n    if device.type == "mps":\n        # PyTorch\'s nested-tensor encoder path is unavailable on MPS for this model.\n        model.transformer.encoder.use_nested_tensor = False\n    model.load_flattened_weights(weights)\n    model.to(device)\n    model.eval()\n    return model\n\n\ndef greedy_decode(\n    model: TranslationTransformer,\n    source_ids: list[int] | torch.Tensor,\n    *,\n    bos_id: int,\n    eos_id: int,\n    pad_id: int,\n    max_len: int = 25,\n    device: torch.device | None = None,\n) -> list[int]:\n    """Decode one source sequence, including BOS and the first EOS when emitted."""\n    if max_len < 1:\n        raise ValueError("max_len must be positive")\n    device = device or next(model.parameters()).device\n    src = torch.as_tensor(source_ids, dtype=torch.long, device=device).reshape(1, -1)\n    generated = torch.tensor([[bos_id]], dtype=torch.long, device=device)\n    model.eval()\n    with torch.no_grad():\n        for _ in range(max_len):\n            logits = model(src, generated, src_pad_idx=pad_id, tgt_pad_idx=pad_id)[:, -1, :]\n            if not bool(torch.isfinite(logits).all()):\n                raise FloatingPointError("non-finite logits during translation decoding")\n            next_id = torch.argmax(logits, dim=-1, keepdim=True)\n            generated = torch.cat([generated, next_id], dim=1)\n            if int(next_id.item()) == eos_id:\n                break\n    return generated[0].detach().cpu().tolist()\n\n\ndef validate_inputs(\n    corpus: Path, split_path: Path, weights_path: Path\n) -> tuple[pd.DataFrame, dict[str, Any], np.ndarray]:\n    split = validate_frozen_split(corpus, split_path)\n    observed_corpus_hash = sha256_file(corpus)\n    if observed_corpus_hash != split["source_corpus_sha256"]:\n        raise ValueError("development split corpus hash does not match the corpus on disk")\n    rows = pd.read_csv(corpus)\n    required = {"id", "asdfghjkl", "english"}\n    if not required.issubset(rows.columns):\n        raise ValueError(f"corpus is missing required columns: {sorted(required - set(rows.columns))}")\n    development_ids = list(split["splits"]["development"])\n    if len(development_ids) != split["counts"]["development"] or len(set(development_ids)) != len(development_ids):\n        raise ValueError("development split has invalid IDs")\n    if set(development_ids) - set(rows["id"]):\n        raise ValueError("development split contains an ID absent from the corpus")\n    selected = rows.set_index("id").loc[development_ids].reset_index()\n    weights = np.load(weights_path, allow_pickle=False)\n    if weights.dtype != np.float32 or weights.size != EXPECTED_PARAMETER_COUNT or not np.isfinite(weights).all():\n        raise ValueError("test noisy weights fail dtype, length, or finite-value validation")\n    if weights_path.resolve() == DEFAULT_WEIGHTS.resolve():\n        manifest = json.loads((weights_path.parent / "manifest.json").read_text(encoding="utf-8"))\n        if sha256_file(weights_path) != manifest["files"][weights_path.name]["sha256"]:\n            raise ValueError("supplied noisy test weights differ from the preflight manifest")\n    return selected, split, weights.reshape(-1)\n\n\ndef evaluate(predictions: list[str], references: list[str]) -> dict[str, Any]:\n    if not references or len(predictions) != len(references):\n        raise ValueError("metrics require equal nonempty prediction/reference lists")\n    bleu = BLEU()\n    bleu_score = bleu.corpus_score(predictions, [references])\n    chrf = CHRF(word_order=2)\n    chrf_score = chrf.corpus_score(predictions, [references])\n    return {\n        "corpus_bleu4": float(bleu_score.score),\n        "chrf_plus_plus": float(chrf_score.score),\n        "exact_match": float(sum(pred == ref for pred, ref in zip(predictions, references)) / len(references)),\n        "metric_versions": {"sacrebleu": sacrebleu.__version__},\n        "metric_signatures": {\n            "bleu": bleu.get_signature().format(),\n            "chrf_plus_plus": chrf.get_signature().format(),\n        },\n    }\n\n\ndef run_baseline(\n    *,\n    notebook: Path = DEFAULT_NOTEBOOK,\n    corpus: Path = DEFAULT_CORPUS,\n    weights: Path = DEFAULT_WEIGHTS,\n    split: Path = DEFAULT_SPLIT,\n    output_json: Path = DEFAULT_JSON,\n    output_csv: Path = DEFAULT_CSV,\n    device_name: str | None = None,\n    max_len: int = 25,\n    checkpoint: Path | None = None,\n) -> dict[str, Any]:\n    rows, split_payload, flat_weights = validate_inputs(corpus, split, weights)\n    source_tokenizer, target_tokenizer = load_tokenizers(notebook)\n    device = torch.device(device_name or ("mps" if torch.backends.mps.is_available() else "cpu"))\n    model = prepare_model(flat_weights, device)\n    checkpoint_progress = None\n    if checkpoint is not None:\n        payload = torch.load(checkpoint, map_location="cpu", weights_only=False)\n        expected_hashes = {\n            "notebook": sha256_file(notebook), "corpus": sha256_file(corpus),\n            "split": sha256_file(split), "weights": sha256_file(weights),\n        }\n        validate_checkpoint(\n            payload, input_hashes=expected_hashes, split_ids=split_payload["splits"],\n            parameter_count=EXPECTED_PARAMETER_COUNT,\n            special_ids={"source_pad": source_tokenizer.pad_id,\n                         "target_pad": target_tokenizer.pad_id,\n                         "target_bos": target_tokenizer.bos_id,\n                         "target_eos": target_tokenizer.eos_id},\n        )\n        model.load_state_dict(payload["model"], strict=True)\n        model.eval()\n        checkpoint_progress = {"epoch": payload.get("epoch"), "step": payload.get("step"), "sha256": sha256_file(checkpoint)}\n    predictions: list[str] = []\n    generated_sequences: list[list[int]] = []\n    for source in rows["asdfghjkl"].astype(str):\n        source_ids = source_tokenizer.encode(source, add_special_tokens=True)\n        generated = greedy_decode(\n            model,\n            source_ids,\n            bos_id=target_tokenizer.bos_id,\n            eos_id=target_tokenizer.eos_id,\n            pad_id=target_tokenizer.pad_id,\n            max_len=max_len,\n            device=device,\n        )\n        predictions.append(target_tokenizer.decode(generated, skip_special_tokens=True))\n        generated_sequences.append(generated)\n    references = rows["english"].astype(str).tolist()\n    metrics = evaluate(predictions, references)\n    # Raw-reference metrics above remain the fixed selection contract. These\n    # diagnostics expose formatting and termination errors without silently\n    # redefining the reported baseline or optimizing on audit references.\n    normalized_references = [" ".join(target_tokenizer.tokenize(ref)) for ref in references]\n    eos_count = sum(target_tokenizer.eos_id in seq[1:] for seq in generated_sequences)\n    token_lists = [prediction.split() for prediction in predictions]\n    repetitive = sum(bool(tokens) and max(tokens.count(token) for token in set(tokens)) / len(tokens) >= 0.5\n                     for tokens in token_lists)\n    per_id = rows[["id", "asdfghjkl", "english"]].copy()\n    per_id["prediction"] = predictions\n    per_id["exact_match"] = per_id["prediction"] == per_id["english"]\n    output_csv.parent.mkdir(parents=True, exist_ok=True)\n    per_id.rename(columns={"asdfghjkl": "source", "english": "reference"}).to_csv(output_csv, index=False)\n    result: dict[str, Any] = {\n        "baseline": ("translation_noisy_weights_no_finetuning" if weights.resolve() == DEFAULT_WEIGHTS.resolve()\n                     else "translation_restored_weights_no_finetuning") if checkpoint is None else "translation_finetuned_weights",\n        "scope": {"split": "development", "rows": len(rows), "ids": rows["id"].tolist()},\n        "metrics": metrics,\n        "diagnostics": {\n            "normalized_reference_metrics": evaluate(predictions, normalized_references),\n            "reference_normalization": "target_tokenizer.tokenize joined with single spaces; diagnostic only",\n            "eos_terminated": eos_count,\n            "cap_terminated": len(predictions) - eos_count,\n            "empty_predictions": sum(not prediction.strip() for prediction in predictions),\n            "unique_predictions": len(set(predictions)),\n            "rows_with_token_share_at_least_half": repetitive,\n            "mean_prediction_tokens": sum(map(len, token_lists)) / len(token_lists),\n            "mean_reference_tokens": sum(len(ref.split()) for ref in normalized_references) / len(references),\n        },\n        "config": {\n            "device": str(device),\n            "max_len_generated_after_bos": max_len,\n            "decoding": "greedy_argmax",\n            "explicit_bos_eos": True,\n            "model_eval": True,\n            "nested_tensor_disabled_on_mps": device.type == "mps",\n            "fine_tuning": checkpoint is not None,\n            "initialization": "supplied_noisy" if weights.resolve() == DEFAULT_WEIGHTS.resolve() else "restored_candidate",\n        },\n        "provenance": {\n            "prediction_csv_sha256": sha256_file(output_csv),\n            "notebook_sha256": sha256_file(notebook),\n            "corpus_sha256": sha256_file(corpus),\n            "split_sha256": sha256_file(split),\n            "weights_sha256": sha256_file(weights),\n            "checkpoint": checkpoint_progress,\n            "split_corpus_hash_validated": True,\n            "weights_file": str(weights),\n            "torch_version": torch.__version__,\n            "python_version": platform.python_version(),\n            "source_vocab_size": source_tokenizer.vocab_size,\n            "target_vocab_size": target_tokenizer.vocab_size,\n            "special_ids": {\n                "source": {"pad": source_tokenizer.pad_id, "bos": source_tokenizer.bos_id, "eos": source_tokenizer.eos_id},\n                "target": {"pad": target_tokenizer.pad_id, "bos": target_tokenizer.bos_id, "eos": target_tokenizer.eos_id},\n            },\n        },\n    }\n    output_json.parent.mkdir(parents=True, exist_ok=True)\n    output_json.write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", type=Path, default=DEFAULT_NOTEBOOK)\n    parser.add_argument("--corpus", type=Path, default=DEFAULT_CORPUS)\n    parser.add_argument("--weights", type=Path, default=DEFAULT_WEIGHTS)\n    parser.add_argument("--split", type=Path, default=DEFAULT_SPLIT)\n    parser.add_argument("--output-json", type=Path, default=DEFAULT_JSON)\n    parser.add_argument("--output-csv", type=Path, default=DEFAULT_CSV)\n    parser.add_argument("--device", dest="device_name", default=None)\n    parser.add_argument("--max-len", type=int, default=25)\n    parser.add_argument("--checkpoint", type=Path)\n    args = parser.parse_args()\n    result = run_baseline(**vars(args))\n    print(json.dumps(result["metrics"], sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'scripts/train_translation.py': '#!/usr/bin/env python3\n"""Bounded fine-tuning for the supplied Team 23 translation Transformer.\n\nThe trainer reads fitting IDs for optimization and development IDs for loss\nvalidation.  It never loads audit rows and does not perform generation or\ncompute translation metrics.  Use ``scripts/run_job.py`` to bound a real run.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\nfrom typing import Any, Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nROOT = Path(__file__).resolve().parents[1]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom scripts.translation_noisy_baseline import (\n    DEFAULT_CORPUS,\n    DEFAULT_NOTEBOOK,\n    DEFAULT_SPLIT,\n    DEFAULT_WEIGHTS,\n    EXPECTED_PARAMETER_COUNT,\n    TranslationTransformer,\n    prepare_model,\n    sha256_file,\n)\nfrom scripts.team23_tokenizer import Team23Tokenizer, load_tokenizers\nfrom scripts.validate_translation_split import validate_frozen_split\n\n\nDEFAULT_OUTPUT = ROOT / "results" / "training" / "translation"\nTRAINER_VERSION = "translation-trainer-v1"\n\n\ndef source_provenance() -> dict[str, Any]:\n    """Return reproducible source identity without requiring Git to be present."""\n    revision: str | None = None\n    git_status = "unavailable"\n    dirty: bool | None = None\n    try:\n        completed = subprocess.run(\n            ["git", "rev-parse", "HEAD"],\n            cwd=ROOT,\n            check=True,\n            capture_output=True,\n            text=True,\n        )\n        revision = completed.stdout.strip() or None\n        if revision is not None:\n            git_status = "available"\n            dirty = (\n                subprocess.run(["git", "diff", "--quiet"], cwd=ROOT, check=False).returncode != 0\n                or subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=ROOT, check=False).returncode != 0\n            )\n    except (OSError, subprocess.CalledProcessError):\n        pass\n    source_files = {}\n    for relative in (\n        "scripts/train_translation.py",\n        "scripts/translation_noisy_baseline.py",\n        "scripts/team23_tokenizer.py",\n        "scripts/validate_translation_split.py",\n        "scripts/translation_checkpoint.py",\n        "scripts/make_translation_split.py",\n    ):\n        path = ROOT / relative\n        if path.is_file():\n            source_files[relative] = sha256_file(path)\n    return {\n        "git_revision": revision,\n        "git_status": git_status,\n        "dirty": dirty,\n        "file_sha256": source_files,\n    }\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef pad_batch(sequences: Sequence[Sequence[int]], pad_id: int, device: torch.device) -> torch.Tensor:\n    if not sequences:\n        raise ValueError("cannot pad an empty batch")\n    width = max(len(sequence) for sequence in sequences)\n    if width < 1:\n        raise ValueError("sequences must contain at least one token")\n    output = torch.full((len(sequences), width), pad_id, dtype=torch.long, device=device)\n    for row, sequence in enumerate(sequences):\n        output[row, : len(sequence)] = torch.as_tensor(sequence, dtype=torch.long, device=device)\n    return output\n\n\ndef shifted_targets(target_ids: Sequence[int], pad_id: int) -> tuple[list[int], list[int]]:\n    """Return decoder input and next-token labels for one BOS/EOS sequence."""\n    if len(target_ids) < 2:\n        raise ValueError("target sequence must contain at least BOS and EOS")\n    decoder_input = list(target_ids[:-1])\n    labels = list(target_ids[1:])\n    if not labels or all(token == pad_id for token in labels):\n        raise ValueError("target labels contain no non-pad token")\n    return decoder_input, labels\n\n\ndef make_batch(\n    examples: Sequence[tuple[Sequence[int], Sequence[int]]],\n    *,\n    src_pad_id: int,\n    tgt_pad_id: int,\n    device: torch.device,\n) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n    if not examples:\n        raise ValueError("cannot make an empty batch")\n    sources = [list(source) for source, _ in examples]\n    decoder_inputs: list[list[int]] = []\n    labels: list[list[int]] = []\n    for _, target in examples:\n        decoder_input, label = shifted_targets(target, tgt_pad_id)\n        decoder_inputs.append(decoder_input)\n        labels.append(label)\n    return (\n        pad_batch(sources, src_pad_id, device),\n        pad_batch(decoder_inputs, tgt_pad_id, device),\n        pad_batch(labels, tgt_pad_id, device),\n    )\n\n\ndef teacher_forced_loss(\n    model: torch.nn.Module,\n    examples: Sequence[tuple[Sequence[int], Sequence[int]]],\n    *,\n    src_pad_id: int,\n    tgt_pad_id: int,\n    device: torch.device,\n) -> torch.Tensor:\n    src, decoder_input, labels = make_batch(\n        examples, src_pad_id=src_pad_id, tgt_pad_id=tgt_pad_id, device=device\n    )\n    logits = model(src, decoder_input, src_pad_idx=src_pad_id, tgt_pad_idx=tgt_pad_id)\n    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=tgt_pad_id)\n    if not torch.isfinite(loss):\n        raise FloatingPointError("non-finite teacher-forced loss")\n    return loss\n\n\ndef iter_batches(\n    examples: Sequence[tuple[Sequence[int], Sequence[int]]],\n    batch_size: int,\n    *,\n    seed: int,\n    epoch: int,\n    shuffle: bool,\n    max_batches: int | None = None,\n) -> Iterable[list[tuple[Sequence[int], Sequence[int]]]]:\n    if batch_size < 1:\n        raise ValueError("batch_size must be positive")\n    order = list(range(len(examples)))\n    if shuffle:\n        generator = torch.Generator(device="cpu")\n        generator.manual_seed(seed + epoch)\n        order = torch.randperm(len(order), generator=generator).tolist()\n    emitted = 0\n    for start in range(0, len(order), batch_size):\n        if max_batches is not None and emitted >= max_batches:\n            break\n        indices = order[start : start + batch_size]\n        yield [examples[index] for index in indices]\n        emitted += 1\n\n\ndef _cpu_tree(value: Any) -> Any:\n    if isinstance(value, torch.Tensor):\n        return value.detach().cpu()\n    if isinstance(value, dict):\n        return {key: _cpu_tree(item) for key, item in value.items()}\n    if isinstance(value, list):\n        return [_cpu_tree(item) for item in value]\n    if isinstance(value, tuple):\n        return tuple(_cpu_tree(item) for item in value)\n    return value\n\n\ndef _validate_finite_tree(value: Any, path: str) -> None:\n    if isinstance(value, torch.Tensor):\n        if not bool(torch.isfinite(value).all()):\n            raise ValueError(f"checkpoint contains non-finite tensor: {path}")\n    elif isinstance(value, float):\n        if not math.isfinite(value):\n            raise ValueError(f"checkpoint contains non-finite value: {path}")\n    elif isinstance(value, dict):\n        for key, item in value.items():\n            _validate_finite_tree(item, f"{path}.{key}")\n    elif isinstance(value, (list, tuple)):\n        for index, item in enumerate(value):\n            _validate_finite_tree(item, f"{path}[{index}]")\n\n\ndef _atomic_torch_save(value: dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)\n    temporary = Path(temporary_name)\n    try:\n        with os.fdopen(fd, "wb") as handle:\n            torch.save(value, handle)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, path)\n        try:\n            directory_fd = os.open(path.parent, os.O_RDONLY)\n            try:\n                os.fsync(directory_fd)\n            finally:\n                os.close(directory_fd)\n        except OSError:\n            pass\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef capture_rng_state() -> dict[str, Any]:\n    state: dict[str, Any] = {\n        "python": random.getstate(),\n        "numpy": np.random.get_state(),\n        "torch": torch.get_rng_state(),\n    }\n    if torch.cuda.is_available():\n        state["cuda"] = torch.cuda.get_rng_state_all()\n    if torch.backends.mps.is_available() and hasattr(torch.mps, "get_rng_state"):\n        state["mps"] = torch.mps.get_rng_state()\n    return state\n\n\ndef restore_rng_state(state: dict[str, Any]) -> None:\n    required = {"python", "numpy", "torch"}\n    if not required.issubset(state):\n        raise ValueError("checkpoint RNG state is incomplete")\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch"])\n    if "cuda" in state and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state["cuda"])\n    elif "cuda" in state:\n        raise ValueError("checkpoint contains CUDA RNG state but CUDA is unavailable")\n    if "mps" in state:\n        if not (torch.backends.mps.is_available() and hasattr(torch.mps, "set_rng_state")):\n            raise ValueError("checkpoint contains MPS RNG state but MPS is unavailable")\n        torch.mps.set_rng_state(state["mps"])\n\n\ndef checkpoint_payload(\n    model: torch.nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\n    *,\n    epoch: int,\n    step: int,\n    config: dict[str, Any],\n    input_hashes: dict[str, str],\n    split_ids: dict[str, list[str]],\n    lineage: dict[str, Any] | None = None,\n    history: list[dict[str, float | int]] | None = None,\n) -> dict[str, Any]:\n    payload = {\n        "contract": TRAINER_VERSION,\n        "model": _cpu_tree(model.state_dict()),\n        "optimizer": _cpu_tree(optimizer.state_dict()),\n        "scheduler": _cpu_tree(scheduler.state_dict()),\n        "epoch": epoch,\n        "step": step,\n        "rng": _cpu_tree(capture_rng_state()),\n        "config": config,\n        "input_hashes": input_hashes,\n        "split_ids": split_ids,\n    }\n    if lineage is not None:\n        payload["lineage"] = lineage\n    if history is not None:\n        payload["history"] = history\n    return payload\n\n\ndef save_checkpoint(payload: dict[str, Any], path: Path) -> None:\n    required = {"contract", "model", "optimizer", "scheduler", "epoch", "step", "rng", "config", "input_hashes", "split_ids"}\n    if not required.issubset(payload) or payload["contract"] != TRAINER_VERSION:\n        raise ValueError("refusing to save an incomplete checkpoint contract")\n    _atomic_torch_save(payload, path)\n\n\ndef load_checkpoint(\n    path: Path,\n    *,\n    model: torch.nn.Module,\n    optimizer: torch.optim.Optimizer,\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\n    config: dict[str, Any],\n    input_hashes: dict[str, str],\n    split_ids: dict[str, list[str]],\n    device: torch.device,\n) -> tuple[int, int]:\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    checkpoint = torch.load(path, map_location="cpu", weights_only=False)\n    required = {"contract", "model", "optimizer", "scheduler", "epoch", "step", "rng", "config", "input_hashes", "split_ids"}\n    if not isinstance(checkpoint, dict) or not required.issubset(checkpoint):\n        raise ValueError("checkpoint is missing required contract fields")\n    if checkpoint["contract"] != TRAINER_VERSION:\n        raise ValueError("checkpoint trainer contract version mismatch")\n    if checkpoint["config"] != config or checkpoint["input_hashes"] != input_hashes or checkpoint["split_ids"] != split_ids:\n        raise ValueError("checkpoint inputs, split IDs, or configuration do not match current run")\n    if (type(checkpoint["epoch"]) is not int or type(checkpoint["step"]) is not int\n            or checkpoint["epoch"] < 0 or checkpoint["step"] < 0):\n        raise ValueError("checkpoint epoch and step must be nonnegative integers")\n    _validate_finite_tree(checkpoint["model"], "model")\n    _validate_finite_tree(checkpoint["optimizer"], "optimizer")\n    _validate_finite_tree(checkpoint["scheduler"], "scheduler")\n    scheduler_state = checkpoint["scheduler"]\n    if (not isinstance(scheduler_state, dict)\n            or type(scheduler_state.get("last_epoch")) is not int\n            or ("_step_count" in scheduler_state and type(scheduler_state["_step_count"]) is not int)\n            or ("_step_count" in scheduler_state and scheduler_state["_step_count"] < 0)):\n        raise ValueError("checkpoint scheduler progress is invalid")\n    model.load_state_dict(checkpoint["model"], strict=True)\n    optimizer.load_state_dict(checkpoint["optimizer"])\n    scheduler.load_state_dict(checkpoint["scheduler"])\n    for state in optimizer.state.values():\n        for key, value in list(state.items()):\n            if isinstance(value, torch.Tensor):\n                state[key] = value.to(device)\n    restore_rng_state(checkpoint["rng"])\n    return checkpoint["epoch"], checkpoint["step"]\n\n\ndef _read_checkpoint(path: Path) -> dict[str, Any]:\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    checkpoint = torch.load(path, map_location="cpu", weights_only=False)\n    if not isinstance(checkpoint, dict):\n        raise ValueError("checkpoint root must be a mapping")\n    return checkpoint\n\n\ndef _assess_history(\n    candidate: Any,\n    *,\n    parent_epoch: int,\n    parent_step: int,\n    source: str,\n) -> tuple[list[dict[str, float | int]], dict[str, Any]]:\n    """Keep only ordered records proven to belong to the parent checkpoint."""\n    records: list[dict[str, float | int]] = []\n    issues: list[str] = []\n    if not isinstance(candidate, list):\n        candidate = []\n        issues.append("history is absent")\n    previous_epoch = 0\n    previous_step = 0\n    for index, record in enumerate(candidate):\n        if not isinstance(record, dict):\n            issues.append(f"record {index} is not an object")\n            continue\n        epoch, step = record.get("epoch"), record.get("step")\n        losses = (record.get("train_loss"), record.get("development_loss"))\n        if type(epoch) is not int or type(step) is not int or epoch < 1 or step < 1:\n            issues.append(f"record {index} has invalid progress")\n            continue\n        if any(not isinstance(loss, (int, float)) or isinstance(loss, bool) or not math.isfinite(float(loss)) for loss in losses):\n            issues.append(f"record {index} has invalid loss")\n            continue\n        if epoch > parent_epoch or step > parent_step:\n            continue\n        if epoch <= previous_epoch or step <= previous_step:\n            issues.append(f"record {index} is out of order")\n            continue\n        records.append(dict(record))\n        previous_epoch, previous_step = epoch, step\n    observed = {int(record["epoch"]) for record in records}\n    missing = [epoch for epoch in range(1, parent_epoch + 1) if epoch not in observed]\n    if missing:\n        issues.append("missing epochs: " + ",".join(str(epoch) for epoch in missing))\n    return records, {"source": source, "complete": not issues, "gaps": issues}\n\n\ndef _parent_history(resume: Path, *, parent_epoch: int, parent_step: int) -> tuple[list[dict[str, float | int]], dict[str, Any]]:\n    """Recover bounded history from a checkpoint, or a legacy sibling summary."""\n    checkpoint = _read_checkpoint(resume)\n    history = checkpoint.get("history")\n    if history is not None:\n        return _assess_history(history, parent_epoch=parent_epoch, parent_step=parent_step, source="checkpoint")\n    summary_path = resume.parent / "summary.json"\n    if summary_path.is_file():\n        try:\n            summary = json.loads(summary_path.read_text(encoding="utf-8"))\n        except (OSError, json.JSONDecodeError):\n            return [], {"source": "summary", "complete": False, "gaps": ["summary is unreadable"]}\n        summary_history = summary.get("history")\n        return _assess_history(summary_history, parent_epoch=parent_epoch, parent_step=parent_step, source="summary")\n    return [], {"source": "none", "complete": parent_epoch == 0, "gaps": [] if parent_epoch == 0 else ["history source is absent"]}\n\n\ndef _parent_summary(resume: Path) -> dict[str, str] | None:\n    summary_path = resume.parent / "summary.json"\n    if not summary_path.is_file():\n        return None\n    return {"path": str(summary_path.resolve()), "sha256": sha256_file(summary_path)}\n\n\ndef prepare_output_dir(output_dir: Path, resume: Path | None) -> None:\n    """Reject ambiguous output reuse while allowing documented continuation."""\n    if not output_dir.exists():\n        output_dir.mkdir(parents=True)\n        return\n    if not output_dir.is_dir():\n        raise FileExistsError(f"output path is not a directory: {output_dir}")\n    entries = {entry.name for entry in output_dir.iterdir()}\n    if not entries:\n        return\n    if resume is not None:\n        try:\n            same_directory = resume.resolve().parent == output_dir.resolve()\n        except OSError:\n            same_directory = False\n        if same_directory and resume.name == "checkpoint_last.pt" and entries <= {"checkpoint_last.pt", "summary.json"}:\n            return\n    raise FileExistsError(\n        f"refusing to overwrite non-empty output directory: {output_dir}; use a fresh directory"\n    )\n\n\ndef validate_resume_target(epochs: int, saved_epoch: int) -> None:\n    if type(saved_epoch) is not int or saved_epoch < 0:\n        raise ValueError("saved checkpoint epoch must be a nonnegative integer")\n    if epochs <= saved_epoch:\n        raise ValueError("epochs must be greater than the saved checkpoint epoch for a continuation")\n\n\ndef _load_data(\n    corpus: Path, split_path: Path, notebook: Path, weights: Path\n) -> tuple[pd.DataFrame, dict[str, Any], Team23Tokenizer, Team23Tokenizer, dict[str, str]]:\n    split = validate_frozen_split(corpus, split_path)\n    if sha256_file(corpus) != split["source_corpus_sha256"]:\n        raise ValueError("split corpus hash does not match corpus")\n    split_ids = {name: list(split["splits"][name]) for name in ("fitting", "development", "audit")}\n    split_sets = {name: set(ids) for name, ids in split_ids.items()}\n    if any(len(split_sets[name]) != len(split_ids[name]) for name in split_ids):\n        raise ValueError("split contains duplicate IDs")\n    if any(split_sets[left] & split_sets[right] for left, right in (("fitting", "development"), ("fitting", "audit"), ("development", "audit"))):\n        raise ValueError("split IDs overlap")\n    selected_ids = split_sets["fitting"] | split_sets["development"]\n    selected_rows: list[dict[str, str]] = []\n    corpus_ids: set[str] = set()\n    with corpus.open("r", encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        required = {"id", "asdfghjkl", "english"}\n        if reader.fieldnames is None or not required.issubset(reader.fieldnames):\n            raise ValueError("corpus is missing required columns")\n        for row in reader:\n            identifier = row["id"]\n            if identifier in corpus_ids:\n                raise ValueError("corpus contains duplicate IDs")\n            corpus_ids.add(identifier)\n            if identifier in selected_ids:\n                if not row["asdfghjkl"] or not row["english"]:\n                    raise ValueError("selected corpus row has an empty source or target")\n                selected_rows.append({key: row[key] for key in required})\n    if set().union(*split_sets.values()) != corpus_ids or len(selected_rows) != len(selected_ids):\n        raise ValueError("split IDs are not exhaustive or a selected row is missing")\n    by_id = pd.DataFrame(selected_rows).set_index("id")\n    source_tokenizer, target_tokenizer = load_tokenizers(notebook)\n    hashes = {\n        "notebook": sha256_file(notebook),\n        "corpus": sha256_file(corpus),\n        "split": sha256_file(split_path),\n        "weights": sha256_file(weights),\n    }\n    return by_id, split, source_tokenizer, target_tokenizer, hashes\n\n\ndef _examples(rows: pd.DataFrame, ids: Sequence[str], source_tokenizer: Team23Tokenizer, target_tokenizer: Team23Tokenizer) -> list[tuple[list[int], list[int]]]:\n    output = []\n    for identifier in ids:\n        row = rows.loc[identifier]\n        output.append((source_tokenizer.encode(str(row["asdfghjkl"])), target_tokenizer.encode(str(row["english"]))))\n    return output\n\n\ndef load_checked_weights(path: Path) -> np.ndarray:\n    values = np.load(path, allow_pickle=False)\n    if values.dtype != np.float32 or values.shape != (EXPECTED_PARAMETER_COUNT,) or not np.isfinite(values).all():\n        raise ValueError("translation initialization must be a finite float32 vector of the exact model length")\n    if path.resolve() == DEFAULT_WEIGHTS.resolve():\n        manifest = json.loads((path.parent / "manifest.json").read_text(encoding="utf-8"))\n        if sha256_file(path) != manifest["files"][path.name]["sha256"]:\n            raise ValueError("supplied noisy test weights differ from the preflight manifest")\n    return values\n\n\ndef run_training(\n    *,\n    notebook: Path = DEFAULT_NOTEBOOK,\n    corpus: Path = DEFAULT_CORPUS,\n    weights: Path = DEFAULT_WEIGHTS,\n    split: Path = DEFAULT_SPLIT,\n    output_dir: Path = DEFAULT_OUTPUT,\n    device_name: str | None = None,\n    epochs: int = 1,\n    batch_size: int = 8,\n    learning_rate: float = 1e-4,\n    seed: int = 2301,\n    smoke: bool = False,\n    smoke_batches: int = 2,\n    resume: Path | None = None,\n) -> dict[str, Any]:\n    if epochs < 1 or batch_size < 1 or learning_rate <= 0 or smoke_batches < 1:\n        raise ValueError("epochs, batch size, smoke batches, and learning rate must be positive")\n    prepare_output_dir(output_dir, resume)\n    rows, split_payload, source_tokenizer, target_tokenizer, hashes = _load_data(corpus, split, notebook, weights)\n    split_ids = {name: list(split_payload["splits"][name]) for name in ("fitting", "development", "audit")}\n    fitting = _examples(rows, split_ids["fitting"], source_tokenizer, target_tokenizer)\n    development = _examples(rows, split_ids["development"], source_tokenizer, target_tokenizer)\n    if smoke:\n        fitting = fitting[: batch_size * smoke_batches]\n    device = torch.device(device_name or ("mps" if torch.backends.mps.is_available() else "cpu"))\n    seed_everything(seed)\n    model = prepare_model(load_checked_weights(weights), device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda _: 1.0)\n    config = {\n        "trainer": TRAINER_VERSION,\n        "architecture": "supplied_TranslationTransformer_v1",\n        "parameter_count": EXPECTED_PARAMETER_COUNT,\n        "batch_size": batch_size,\n        "learning_rate": learning_rate,\n        "seed": seed,\n        "smoke": smoke,\n        "smoke_batches": smoke_batches,\n        "device": str(device),\n        "special_ids": {"source_pad": source_tokenizer.pad_id, "target_pad": target_tokenizer.pad_id, "target_bos": target_tokenizer.bos_id, "target_eos": target_tokenizer.eos_id},\n        "nested_tensor_disabled_on_mps": device.type == "mps",\n    }\n    start_epoch, step = 0, 0\n    checkpoint_path = output_dir / "checkpoint_last.pt"\n    source = source_provenance()\n    parent_checkpoint: dict[str, Any] | None = None\n    parent_summary: dict[str, str] | None = None\n    history: list[dict[str, float | int]] = []\n    history_status: dict[str, Any] = {"source": "new-run", "complete": True, "gaps": []}\n    if resume is not None:\n        parent_payload = _read_checkpoint(resume)\n        if type(parent_payload.get("step")) is not int or parent_payload["step"] < 0:\n            raise ValueError("parent checkpoint epoch and step must be valid integers")\n        validate_resume_target(epochs, parent_payload.get("epoch"))\n        parent_checkpoint = {\n            "path": str(resume.resolve()),\n            "sha256": sha256_file(resume),\n            "epoch": parent_payload["epoch"],\n            "step": parent_payload["step"],\n        }\n        parent_summary = _parent_summary(resume)\n        history, history_status = _parent_history(\n            resume,\n            parent_epoch=parent_payload["epoch"],\n            parent_step=parent_payload["step"],\n        )\n        start_epoch, step = load_checkpoint(resume, model=model, optimizer=optimizer, scheduler=scheduler, config=config, input_hashes=hashes, split_ids=split_ids, device=device)\n    model.train()\n    run_history: list[dict[str, float | int]] = []\n    lineage = {\n        "source": source,\n        "parent_checkpoint": parent_checkpoint,\n        "parent_summary": parent_summary,\n        "history": history_status,\n    }\n    for epoch in range(start_epoch, epochs):\n        train_loss = 0.0\n        train_batches = 0\n        for batch in iter_batches(fitting, batch_size, seed=seed, epoch=epoch, shuffle=True, max_batches=smoke_batches if smoke else None):\n            optimizer.zero_grad(set_to_none=True)\n            loss = teacher_forced_loss(model, batch, src_pad_id=source_tokenizer.pad_id, tgt_pad_id=target_tokenizer.pad_id, device=device)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            scheduler.step()\n            train_loss += float(loss.detach().cpu())\n            train_batches += 1\n            step += 1\n        if not train_batches:\n            raise ValueError("fitting split produced no training batches")\n        model.eval()\n        with torch.no_grad():\n            validation_loss = float(teacher_forced_loss(model, development, src_pad_id=source_tokenizer.pad_id, tgt_pad_id=target_tokenizer.pad_id, device=device).cpu())\n        if not np.isfinite(validation_loss):\n            raise FloatingPointError("non-finite development validation loss")\n        model.train()\n        record = {"epoch": epoch + 1, "step": step, "train_loss": train_loss / train_batches, "development_loss": validation_loss}\n        history.append(record)\n        run_history.append(record)\n        save_checkpoint(checkpoint_payload(model, optimizer, scheduler, epoch=epoch + 1, step=step, config=config, input_hashes=hashes, split_ids=split_ids, lineage=lineage, history=history), checkpoint_path)\n    result = {"trainer": TRAINER_VERSION, "epochs_completed": epochs, "step": step, "history": history, "run_history": run_history, "lineage": lineage, "checkpoint": str(checkpoint_path), "scope": {"fitting_ids": len(fitting), "development_ids": len(development), "audit_ids_loaded": 0}}\n    (output_dir / "summary.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n    return result\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", type=Path, default=DEFAULT_NOTEBOOK)\n    parser.add_argument("--corpus", type=Path, default=DEFAULT_CORPUS)\n    parser.add_argument("--weights", type=Path, default=DEFAULT_WEIGHTS)\n    parser.add_argument("--split", type=Path, default=DEFAULT_SPLIT)\n    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)\n    parser.add_argument("--device", dest="device_name")\n    parser.add_argument("--epochs", type=int, default=1)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--learning-rate", type=float, default=1e-4)\n    parser.add_argument("--seed", type=int, default=2301)\n    parser.add_argument("--smoke", action="store_true")\n    parser.add_argument("--smoke-batches", type=int, default=2)\n    parser.add_argument("--resume", type=Path)\n    args = parser.parse_args()\n    print(json.dumps(run_training(**vars(args)), sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
SOURCE_BUNDLE_SHA256 = '591943802b6e80238ba93a101b8620a1aac55ad3e26888e05b85d93012451f33'
REQUIRED_FILES = ['boilerplate.ipynb', 'train_weights_clean.npy', 'train_weights_noisy.npy', 'test_weights_noisy.npy', 'train_parallel_corpus.csv', 'test_queries.csv']
PINNED_INPUT_HASHES = {'boilerplate.ipynb': 'd99a021e8747bb9b69248ed586c664eda02fb56aa1fa5472e287b770c5a73d6c', 'train_weights_clean.npy': 'aa8c6c8b743fc68cc0fa6dd7bb7a9690bbeace57894beec81d8f1d56a044edc4', 'train_weights_noisy.npy': 'e5292a4b2f0c43e789f7bb2558d98fc2f11b7e9734eed28e77a78efddeae395a', 'test_weights_noisy.npy': 'b751ffee729b96418148a7d7c684d3e8cc08ba46f0e225991f7a3811eb754397', 'train_parallel_corpus.csv': '39f3fd2d9bfb2804e4643072d515af054f1f4c34dbc05f05fea04e9d5e167171', 'test_queries.csv': '30ec797c5769b6a4e233a26aeaa3c38453e8645cc5db3045fcd08dc2a690e32f'}
ROSTER_TEXT = 'Team 23 submission metadata is supplied privately at delivery time. Completed contributions and official membership must be confirmed by the team; no personal names, SRNs, or contribution assignments are stored in this shared repository.'
EXPECTED_WEIGHT_COUNT = 2_961_630
EXPECTED_TRAIN_COUNT = 5_923_260

def _sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _install_source_bundle():
    runtime = Path(tempfile.mkdtemp(prefix="team23_runtime_"))
    package = runtime / "scripts"
    package.mkdir()
    (package / "__init__.py").write_text("", encoding="utf-8")
    for relative, code in SOURCE_BUNDLE.items():
        destination = runtime / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(code, encoding="utf-8")
    sys.path.insert(0, str(runtime))
    return runtime

def _candidate_dirs():
    explicit = os.environ.get("TEAM23_DATA_DIR")
    if explicit:
        path = Path(explicit)
        if path.name != "23_Team_Toxic":
            raise RuntimeError("TEAM23_DATA_DIR must name a 23_Team_Toxic directory")
        return [path]
    roots = [Path.cwd(), Path("/kaggle/input")]
    candidates = []
    for root in roots:
        if not root.is_dir():
            continue
        if root.name == "23_Team_Toxic":
            candidates.append(root)
            continue
        for current, dirs, _files in os.walk(root):
            current_path = Path(current)
            if current_path.name == "23_Team_Toxic":
                candidates.append(current_path)
                dirs[:] = []
            else:
                candidates.extend(current_path / name for name in dirs if name == "23_Team_Toxic")
                dirs[:] = [name for name in dirs if name != "23_Team_Toxic"]
    return candidates

def discover_team23_runtime():
    candidates = {p.resolve() for p in _candidate_dirs() if p.exists()}
    if len(candidates) != 1:
        raise RuntimeError("Team 23 discovery expected exactly one 23_Team_Toxic directory; candidates=" + repr(sorted(map(str, candidates))))
    data_dir = next(iter(candidates))
    manifest_path = data_dir / "manifest.json"
    manifest = None
    manifest_status = "manifest_unavailable"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("team") != 23 or set(manifest.get("required_files", ())) != set(REQUIRED_FILES) or len(manifest.get("required_files", ())) != len(REQUIRED_FILES):
            raise RuntimeError("Team 23 manifest identity or required file list is invalid")
        manifest_status = "verified"
    hashes = {}
    for name in REQUIRED_FILES:
        path = data_dir / name
        if not path.is_file():
            raise RuntimeError("missing Team 23 input: " + name)
        hashes[name] = _sha256(path)
        if manifest is not None and hashes[name] != manifest.get("files", {}).get(name, {}).get("sha256"):
            raise RuntimeError("manifest hash mismatch: " + name)
        if hashes[name] != PINNED_INPUT_HASHES[name]:
            raise RuntimeError("pinned Team 23 hash mismatch: " + name)
    runtime_manifest = runtime_root / "private_runtime_manifest.json"
    runtime_manifest.write_text(json.dumps({"team": 23, "required_files": REQUIRED_FILES, "files": {name: {"sha256": digest} for name, digest in hashes.items()}}, sort_keys=True) + "\n", encoding="utf-8")
    return data_dir, {"team": 23, "manifest": str(manifest_path) if manifest is not None else manifest_status, "manifest_status": manifest_status, "runtime_manifest": str(runtime_manifest), "input_sha256": hashes}

def _stage_runtime_data(data_dir):
    staged = runtime_root / "data" / "23_Team_Toxic"
    staged.mkdir(parents=True, exist_ok=True)
    files = {}
    for name in REQUIRED_FILES:
        destination = staged / name
        if not destination.exists():
            destination.symlink_to(Path(data_dir) / name)
        files[name] = {"sha256": _sha256(destination)}
    (staged / "manifest.json").write_text(json.dumps({"team": 23, "required_files": REQUIRED_FILES, "files": files}, sort_keys=True) + "\n", encoding="utf-8")
    return staged

def _output_dir():
    chosen = os.environ.get("TEAM23_OUTPUT_DIR")
    if chosen:
        return Path(chosen).resolve()
    return (Path("/kaggle/working") if Path("/kaggle").is_dir() else Path.cwd() / "output").resolve()

def _fresh_output(path, protected, *, forbidden_names=(), require_empty=False):
    path = Path(path).resolve()
    if any(path == p or p in path.parents for p in protected):
        raise RuntimeError("output path is inside protected raw input or checkpoint location")
    path.mkdir(parents=True, exist_ok=True)
    collisions = [path / name for name in forbidden_names if (path / name).exists()]
    if collisions:
        raise RuntimeError("refusing to overwrite existing target artifacts: " + repr([str(item) for item in collisions]))
    if require_empty and any(path.iterdir()):
        raise RuntimeError("training output directory must initially be empty: " + str(path))
    return path

def _versions():
    import numpy, pandas, torch, sacrebleu
    return {"python": platform.python_version(), "numpy": numpy.__version__, "pandas": pandas.__version__, "torch": torch.__version__, "sacrebleu": sacrebleu.__version__}

runtime_root = _install_source_bundle()
try:
    import numpy as np
    import pandas as pd
    import torch
    import sacrebleu
except ImportError as exc:
    raise RuntimeError("required Kaggle runtime dependency is unavailable; no installation is attempted") from exc
print("Team 23 | source bundle SHA256:", SOURCE_BUNDLE_SHA256)
print(ROSTER_TEXT)
print("library versions:", json.dumps(_versions(), sort_keys=True))


In [ ]:
data_dir, identity = discover_team23_runtime()
part1_dir = Path(os.environ.get("TEAM23_PART1_DIR", str(_output_dir()))).resolve()
if not part1_dir.is_dir():
    raise RuntimeError("TEAM23_PART1_DIR must point to the restored Part 1 handoff directory")
part1_npy = part1_dir / "denoised_test_weights.npy"
part1_sidecar = part1_dir / "part1_artifact_metadata.json"
if not part1_npy.is_file() or not part1_sidecar.is_file():
    raise RuntimeError("Part 2 requires the exact Part 1 NPY and metadata sidecar; noisy fallback is forbidden")
part1 = json.loads(part1_sidecar.read_text(encoding="utf-8"))
queries_path, corpus_path = data_dir / "test_queries.csv", data_dir / "train_parallel_corpus.csv"
queries = pd.read_csv(queries_path, dtype=str, keep_default_na=False)
if list(queries.columns) != ["id", "asdfghjkl"] or len(queries) != 200 or queries["id"].duplicated().any():
    raise RuntimeError("Team 23 query contract failed")
print("Part 1 handoff candidate:", part1_npy)


In [ ]:
from scripts.train_scalar_mlp import split_labels
expected_scalar_config = {"hidden": 32, "learning_rate": 1e-3, "weight_decay": 0.0, "batch_size": 4096, "epochs": 2, "chunk_size": 65536, "seed": 2301}
expected_input_hashes = {"clean": identity["input_sha256"]["train_weights_clean.npy"], "noisy": identity["input_sha256"]["train_weights_noisy.npy"]}
labels_for_gate, split_for_gate = split_labels(EXPECTED_TRAIN_COUNT, seed=2301, block_size=4096, guard=32)
split_for_gate["labels_sha256"] = hashlib.sha256(labels_for_gate.tobytes()).hexdigest()

def validate_part1_handoff(sidecar, npy_path, csv_path):
    if sidecar.get("team") != 23 or sidecar.get("title") != "AFML Orange Problem — Team 23":
        raise RuntimeError("Part 1 sidecar team identity is invalid")
    if sidecar.get("source_bundle_sha256") != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Part 1 source bundle hash differs from this notebook")
    if sidecar.get("schema_version") != 2 or sidecar.get("method") != "schema2_scalar_residual_mlp":
        raise RuntimeError("Part 1 schema or method is not the fixed scalar recipe")
    if sidecar.get("shape") != [EXPECTED_WEIGHT_COUNT] or sidecar.get("dtype") != "float32" or sidecar.get("submission_rows") != EXPECTED_WEIGHT_COUNT:
        raise RuntimeError("Part 1 shape, dtype, or row-count metadata is invalid")
    if sidecar.get("input_sha256") != expected_input_hashes:
        raise RuntimeError("Part 1 clean/noisy input hashes do not match Team 23")
    if sidecar.get("test_noisy_sha256") != identity["input_sha256"]["test_weights_noisy.npy"]:
        raise RuntimeError("Part 1 test noisy input hash does not match Team 23")
    if sidecar.get("config") != expected_scalar_config:
        raise RuntimeError("Part 1 scalar configuration is not the exact seven-field fixed recipe")
    if sidecar.get("split_sha256") != split_for_gate["labels_sha256"] or sidecar.get("split_spec_sha256") != split_for_gate["spec_sha256"]:
        raise RuntimeError("Part 1 scalar split hash or specification is invalid")
    if sidecar.get("restored_npy_sha256") != _sha256(npy_path):
        raise RuntimeError("Part 1 restored NPY hash does not match its sidecar")
    if not csv_path.is_file() or sidecar.get("submission_csv_sha256") != _sha256(csv_path):
        raise RuntimeError("Part 1 CSV hash is missing or stale")
    return sidecar["restored_npy_sha256"]

restored_hash = validate_part1_handoff(part1, part1_npy, part1_dir / "submission_part1.csv")
weights = np.load(part1_npy, allow_pickle=False)
if weights.dtype != np.dtype("float32") or weights.shape != (EXPECTED_WEIGHT_COUNT,) or not np.isfinite(weights).all():
    raise RuntimeError("restored Part 1 vector fails exact dtype/count/finite contract")
output_dir = _fresh_output(_output_dir(), {data_dir.resolve()}, forbidden_names=("submission_part2.csv", "part2_artifact_metadata.json"))
print("Part 1 initialization hash:", restored_hash)


In [ ]:
from scripts.make_translation_split import make_split, write_json
from scripts.validate_translation_split import validate_frozen_split
from scripts.team23_tokenizer import load_tokenizers
from scripts.translation_noisy_baseline import EXPECTED_PARAMETER_COUNT, prepare_model, greedy_decode
from scripts.train_translation import run_training

runtime_data = runtime_root / "data" / "23_Team_Toxic"
runtime_data.mkdir(parents=True, exist_ok=True)
boilerplate_link = runtime_data / "boilerplate.ipynb"
boilerplate_link.symlink_to(data_dir / "boilerplate.ipynb")
corpus_link = runtime_data / "train_parallel_corpus.csv"
corpus_link.symlink_to(corpus_path)
split_path = runtime_data / "translation_split.json"
write_json(split_path, make_split(corpus_path, seed=2301, expected_rows=300))
split = validate_frozen_split(corpus_path, split_path)
if split["counts"] != {"fitting": 210, "development": 45, "audit": 45}:
    raise RuntimeError("frozen source-only translation split counts failed")
source_tok, target_tok = load_tokenizers(boilerplate_link)
if {"<pad>": source_tok.pad_id, "<bos>": source_tok.bos_id, "<eos>": source_tok.eos_id, "<unk>": source_tok.unk_id} != {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}:
    raise RuntimeError("supplied Team 23 token IDs failed")


In [ ]:
training_candidate = Path(os.environ.get("TEAM23_PART2_TRAINING_DIR", str(output_dir / "part2_training"))).resolve()
if training_candidate in {part1_dir, data_dir}:
    raise RuntimeError("Part 2 training directory cannot be the Part 1 handoff or raw input directory")
training_dir = _fresh_output(training_candidate, {data_dir.resolve()}, require_empty=True)
device_name = os.environ.get("TEAM23_TRANSLATION_DEVICE") or ("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
result = run_training(notebook=boilerplate_link, corpus=corpus_path, weights=part1_npy, split=split_path, output_dir=training_dir, device_name=device_name, epochs=20, batch_size=8, learning_rate=1e-4, seed=2301, smoke=False)
checkpoint_path = Path(result["checkpoint"])
if result.get("scope", {}).get("audit_ids_loaded") != 0:
    raise RuntimeError("audit English must remain reserved")


In [ ]:
checkpoint_hash = _sha256(checkpoint_path)
payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model = prepare_model(weights, torch.device(device_name))
model.load_state_dict(payload["model"], strict=True)
model.eval()
rows = []
with queries_path.open("r", encoding="utf-8-sig", newline="") as handle:
    for row in csv.DictReader(handle):
        source_ids = source_tok.encode(row["asdfghjkl"])
        decoded = greedy_decode(model, source_ids, bos_id=target_tok.bos_id, eos_id=target_tok.eos_id, pad_id=target_tok.pad_id, max_len=25, device=torch.device(device_name))
        text_value = target_tok.decode(decoded).strip()
        if not text_value or any(token in text_value for token in ("<pad>", "<bos>", "<eos>", "<unk>")):
            raise RuntimeError("decoded query contains empty text or a special-token string")
        rows.append((row["id"], text_value))
if [row[0] for row in rows] != queries["id"].astype(str).tolist() or len(rows) != 200:
    raise RuntimeError("Part 2 query ID order/count failed")
submission_path = output_dir / "submission_part2.csv"
with submission_path.open("x", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle, lineterminator="\n")
    writer.writerow(("id", "translation"))
    writer.writerows(rows)
metadata = {"team": 23, "title": "AFML Orange Problem — Team 23", "method": "supplied_TranslationTransformer_v1_modified_tokenizer_adapter", "source_bundle_sha256": SOURCE_BUNDLE_SHA256, "boilerplate_sha256": _sha256(boilerplate_link), "initialization_sha256": restored_hash, "checkpoint_sha256": checkpoint_hash, "prediction_sha256": _sha256(submission_path), "split_sha256": _sha256(split_path), "split_counts": split["counts"], "config": payload["config"], "epochs_completed": result["epochs_completed"], "audit_metrics": None, "roster": ROSTER_TEXT, "contributions": "PRIVATE_DELIVERY_METADATA_PENDING"}
(output_dir / "part2_artifact_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("Part 2 outputs:", submission_path, output_dir / "part2_artifact_metadata.json")


In [ ]:
with (output_dir / "submission_part2.csv").open("r", encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle)
    if next(reader, None) != ["id", "translation"]:
        raise RuntimeError("Part 2 CSV header mismatch")
    checked = list(reader)
if len(checked) != 200 or [row[0] for row in checked] != queries["id"].astype(str).tolist() or any(len(row) != 2 or not row[1].strip() for row in checked):
    raise RuntimeError("Part 2 CSV exact output contract failed")
print("Part 2 schema/hash validation passed; no audit was computed or retained.")
